In [ ]:
#@title Cell 30.1 - Notebook overview
# This cell states the purpose, fixed model boundary and expected outputs of Notebook 30.

from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 30: MIC Prediction for a New E. coli Pathogen Using Model C

## Purpose

Notebook 30 will apply the completed Model C reference model to one new
*Escherichia coli* pathogen with no measured MIC values.

It will:

1. accept one NCBI BioSample accession and identify its assembly accession;
2. construct the same 267, 833, 25 and five Model 3B pathogen features;
3. calculate the new pathogen's Model 3B similarity to the 9,058 Model C
   training pathogens;
4. extract the same 297 targeted AMR-related loci used in Notebook 23;
5. select one accepted query sequence per locus using the Notebook 24 rule;
6. add each query sequence to the corresponding fixed Notebook 24 multiple
   alignment without changing the training alignment positions;
7. calculate the selected locus-weighted sequence-similarity vector;
8. combine the two similarity vectors using

\[
K_P^{(C)}(q,j)
=0.4K_{\mathrm{seq}}(q,j)
+0.6K_P^{(3B)}(q,j);
\]

9. calculate the nearest-training-pathogen similarity
   \(\max_j K_P^{(C)}(q,j)\);
10. project the new pathogen into the fixed 256-coordinate Model C
    representation;
11. predict MIC for all 26 antibiotics;
12. attach antibiotic-specific 95% prediction limits and the empirical
    prediction-support status; and
13. save and package the complete query record, sequence checks and MIC table.

## Fixed model boundary

Notebook 30 will not retrain Model C or alter any Notebook 23, 24, 26, 27 or
28 output. The selected Model C settings remain:

- locus-weighted sequence kernel;
- sequence contribution \(\rho=0.4\);
- 256 pathogen coordinates;
- 26 antibiotic coordinates; and
- Ridge penalty \(\alpha=1\).

A missing or unreliable query sequence is not treated as an alignment gap.
Query nucleotides that cannot be placed in the fixed training alignment are
reported and do not change the training alignment positions.

The predicted MIC values and 95% prediction limits are research outputs. They
are not clinical breakpoints or treatment recommendations.

## Expected notebook length

Notebook 30 contains **16 cells**.
"""))

print(
    "Transition: Cell 30.2 will import the required packages, mount "
    "Google Drive and define the fixed Notebook 30 settings."
)


In [ ]:
#@title Cell 30.2 - Import packages and define notebook settings
# This cell imports the required packages, connects Google Drive and defines all fixed dimensions, filenames and directories.

from pathlib import Path
from datetime import datetime, timezone
import gzip
import hashlib
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
import zipfile

if importlib.util.find_spec("Bio") is None:
    print("Installing Biopython...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "biopython>=1.83,<2"],
        check=True,
    )

import joblib
import numpy as np
import pandas as pd
from Bio import SeqIO
from IPython.display import display
from scipy import sparse
from google.colab import drive

drive.mount("/content/drive")

EXPECTED_MODEL3B_PATHOGENS = 9_377
EXPECTED_MODEL_C_PATHOGENS = 9_058
EXPECTED_ANTIBIOTICS = 26
EXPECTED_TARGETED_LOCI = 297
EXPECTED_REPRESENTED_LOCI = 265
EXPECTED_FULL_GENE_LOCI = 266
EXPECTED_CODING_REFERENCE_LOCI = 30

EXPECTED_FULL_GENE_FEATURES = 267
EXPECTED_CORE_DETERMINANT_FEATURES = 833
EXPECTED_POINT_TARGET_FEATURES = 25
EXPECTED_BURDEN_FEATURES = 5

EXPECTED_PATHOGEN_COORDINATES = 256
EXPECTED_ANTIBIOTIC_COORDINATES = 26
EXPECTED_INTERACTION_FEATURES = 6_656

FULL_GENE_WEIGHT = 0.30
CORE_DETERMINANT_WEIGHT = 0.30
POINT_TARGET_WEIGHT = 0.20
BURDEN_WEIGHT = 0.20

SELECTED_SEQUENCE_KERNEL = "locus-weighted"
SELECTED_RHO = 0.4
SELECTED_RIDGE_ALPHA = 1.0
EXPECTED_AMRFINDERPLUS_VERSION = "4.2.7"
EXPECTED_AMRFINDER_DATABASE_VERSION = "2026-05-15.1"

THREADS = 2
COMMAND_RETRIES = 3
MINIMUM_NUCLEOTIDE_IDENTITY = 90.0
MINIMUM_QUERY_COVERAGE = 90.0
OMPC_MINIMUM_NUCLEOTIDE_IDENTITY = 85.0
OMPC_MINIMUM_QUERY_COVERAGE = 99.0
BORDERLINE_IDENTITY = 80.0
BORDERLINE_COVERAGE = 80.0

PROJECT_DIRECTORY = Path("/content/drive/MyDrive/Model3_MIC_Project")
NOTEBOOK23_DIRECTORY = PROJECT_DIRECTORY / "notebook23"
NOTEBOOK24_DIRECTORY = PROJECT_DIRECTORY / "notebook24"
NOTEBOOK24_RESULT_DIRECTORY = NOTEBOOK24_DIRECTORY / "results"
NOTEBOOK24_ALIGNMENT_DIRECTORY = NOTEBOOK24_DIRECTORY / "alignment_checkpoints"
NOTEBOOK30_DIRECTORY = PROJECT_DIRECTORY / "notebook30"
NOTEBOOK30_RESULT_DIRECTORY = NOTEBOOK30_DIRECTORY / "results"
NOTEBOOK30_CHECKPOINT_DIRECTORY = NOTEBOOK30_DIRECTORY / "query_alignment_checkpoints"

WORK_DIRECTORY = Path("/content/notebook30_work")
INPUT_DIRECTORY = WORK_DIRECTORY / "inputs"
ASSEMBLY_WORK_DIRECTORY = WORK_DIRECTORY / "query_assembly"
ALIGNMENT_WORK_DIRECTORY = WORK_DIRECTORY / "query_alignments"

for directory in [
    NOTEBOOK30_DIRECTORY,
    NOTEBOOK30_RESULT_DIRECTORY,
    NOTEBOOK30_CHECKPOINT_DIRECTORY,
    WORK_DIRECTORY,
    INPUT_DIRECTORY,
    ASSEMBLY_WORK_DIRECTORY,
    ALIGNMENT_WORK_DIRECTORY,
]:
    directory.mkdir(parents=True, exist_ok=True)

KERNEL15B_ARCHIVE_NAME = "15B_full_gene_genomic_antibiotic_kernels_outputs.zip"
MODEL26_ARCHIVE_NAME = "26_model_c_nested_pathogen_out_and_reference_model_outputs.zip"
SUPPORT27_ARCHIVE_NAME = "27_model_c_pathogen_similarity_support_outputs.zip"
INTERVAL28_ARCHIVE_NAME = "28_model_c_biosample_level_mic_prediction_outputs.zip"

# Set this only if more than one fixed NCBI AMR metadata release is stored in MyDrive.
NCBI_METADATA_PATH_OVERRIDE = ""

def file_sha256(file_path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(file_path, "rb") as input_file:
        for block in iter(lambda: input_file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

settings_summary = pd.DataFrame([
    {"setting": "Model C training pathogens", "value": EXPECTED_MODEL_C_PATHOGENS},
    {"setting": "Targeted AMR-related loci", "value": EXPECTED_TARGETED_LOCI},
    {"setting": "Represented training loci", "value": EXPECTED_REPRESENTED_LOCI},
    {"setting": "Sequence kernel", "value": SELECTED_SEQUENCE_KERNEL},
    {"setting": "Sequence contribution rho", "value": SELECTED_RHO},
    {"setting": "Pathogen coordinates", "value": EXPECTED_PATHOGEN_COORDINATES},
    {"setting": "Antibiotics", "value": EXPECTED_ANTIBIOTICS},
    {"setting": "Notebook 30 output directory", "value": str(NOTEBOOK30_DIRECTORY)},
])
display(settings_summary)

print("Notebook 30 packages, directories and fixed settings were defined.")
print("\nTransition: Cell 30.3 will locate and validate every fixed input.")


In [ ]:
#@title Cell 30.3 - Locate and validate the fixed Model C inputs
# This cell locates the validated Model 3B, Model C, prediction-support and prediction-interval files without modifying them.

def locate_unique_file(file_name):
    candidates = sorted(PROJECT_DIRECTORY.rglob(file_name))
    if not candidates:
        raise FileNotFoundError(
            f"{file_name} was not found under {PROJECT_DIRECTORY}."
        )
    if len(candidates) == 1:
        return candidates[0]
    hashes = {file_sha256(path) for path in candidates}
    if len(hashes) != 1:
        raise ValueError(
            f"Multiple different files named {file_name} were found: {candidates}"
        )
    print(f"Multiple identical copies of {file_name} were found. Using {candidates[0]}")
    return candidates[0]


def archive_member_by_basename(archive, required_name):
    matches = [
        member for member in archive.namelist()
        if Path(member).name == required_name
    ]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one archive member named {required_name}; observed {matches}."
        )
    return matches[0]


def extract_required_members(archive_path, destination_directory, required_names):
    destination_directory.mkdir(parents=True, exist_ok=True)
    extracted = {}
    with zipfile.ZipFile(archive_path, "r") as archive:
        damaged_member = archive.testzip()
        if damaged_member is not None:
            raise ValueError(f"Damaged archive member: {damaged_member}")
        for required_name in required_names:
            member = archive_member_by_basename(archive, required_name)
            output_path = destination_directory / required_name
            with archive.open(member) as source_file, open(output_path, "wb") as output_file:
                shutil.copyfileobj(source_file, output_file)
            extracted[required_name] = output_path
    return extracted


def validate_files_against_manifest(manifest_path, extracted_files):
    with open(manifest_path, "r", encoding="utf-8") as input_file:
        manifest = json.load(input_file)
    if manifest.get("validation_status") != "passed":
        raise ValueError(f"{manifest_path.name} does not report passed validation.")
    records = {record["file_name"]: record for record in manifest.get("files", [])}
    for file_name, path in extracted_files.items():
        if file_name.endswith("output_manifest.json"):
            continue
        if file_name not in records:
            raise ValueError(f"{file_name} is not listed in {manifest_path.name}.")
        if file_sha256(path) != records[file_name]["sha256"]:
            raise ValueError(f"Checksum mismatch for {file_name}.")
    return manifest


notebook23_required_files = {
    "locus_panel": NOTEBOOK23_DIRECTORY / "23_target_amr_locus_panel.csv",
    "reference_audit": NOTEBOOK23_DIRECTORY / "23_target_amr_reference_audit.csv",
}

notebook24_required_files = {
    "available_locus_counts": NOTEBOOK24_RESULT_DIRECTORY / "24_available_locus_counts.npy",
    "alignment_validation": NOTEBOOK24_RESULT_DIRECTORY / "24_alignment_validation.csv",
    "pathogen_order": NOTEBOOK24_RESULT_DIRECTORY / "24_model_c_pathogen_order.csv",
    "locus_kernel_metadata": NOTEBOOK24_RESULT_DIRECTORY / "24_sequence_kernel_locus_weighted_metadata.json",
}

for input_path in [*notebook23_required_files.values(), *notebook24_required_files.values()]:
    if not input_path.exists():
        raise FileNotFoundError(f"Required fixed input was not found: {input_path}")

if not NOTEBOOK24_ALIGNMENT_DIRECTORY.exists():
    raise FileNotFoundError(
        f"Notebook 24 alignment checkpoints were not found: {NOTEBOOK24_ALIGNMENT_DIRECTORY}"
    )

archive_paths = {
    "Notebook 15B": locate_unique_file(KERNEL15B_ARCHIVE_NAME),
    "Notebook 26": locate_unique_file(MODEL26_ARCHIVE_NAME),
    "Notebook 27": locate_unique_file(SUPPORT27_ARCHIVE_NAME),
    "Notebook 28": locate_unique_file(INTERVAL28_ARCHIVE_NAME),
}

files15b = extract_required_members(
    archive_paths["Notebook 15B"],
    INPUT_DIRECTORY / "notebook15B",
    [
        "15B_pathogen_full_gene_representation.npz",
        "15B_pathogen_core_determinant_representation.npz",
        "15B_pathogen_point_target_representation.npz",
        "15B_pathogen_burden_representation.npz",
        "15B_pathogen_preprocessing.npz",
        "15B_pathogen_kernel_index.csv",
        "15B_kernel_configuration.csv",
    ],
)

required26 = [
    "26_output_manifest.json",
    "26_model_c_aligned_interactions.csv.gz",
    "26_final_model_c_reference_model.joblib",
    "26_final_model_c_embeddings.npz",
    "26_final_model_c_pathogen_embedding_index.csv",
    "26_final_model_c_antibiotic_embedding_index.csv",
    "26_final_model_c_configuration.json",
]
files26 = extract_required_members(
    archive_paths["Notebook 26"], INPUT_DIRECTORY / "notebook26", required26
)
manifest26 = validate_files_against_manifest(files26["26_output_manifest.json"], files26)

required27 = [
    "27_output_manifest.json",
    "27_model_c_prediction_support_configuration.json",
]
files27 = extract_required_members(
    archive_paths["Notebook 27"], INPUT_DIRECTORY / "notebook27", required27
)
manifest27 = validate_files_against_manifest(files27["27_output_manifest.json"], files27)

required28 = [
    "28_output_manifest.json",
    "28_model_c_antibiotic_prediction_interval_calibration.csv",
]
files28 = extract_required_members(
    archive_paths["Notebook 28"], INPUT_DIRECTORY / "notebook28", required28
)
manifest28 = validate_files_against_manifest(files28["28_output_manifest.json"], files28)

input_summary = pd.DataFrame([
    {
        "input": name,
        "path": str(path),
        "size_MB": round(path.stat().st_size / (1024 ** 2), 2),
    }
    for name, path in archive_paths.items()
])
display(input_summary)

print("All fixed input files and archives were located and validated.")
print("\nTransition: Cell 30.4 will load the fixed representations, align their row order and validate the final model settings.")


In [ ]:
#@title Cell 30.4 - Load and align the fixed reference representations
# This cell loads the Model 3B representations, Model C embeddings, sequence-alignment support and final prediction settings.

with np.load(files15b["15B_pathogen_preprocessing.npz"], allow_pickle=True) as archive:
    preprocessing = {name: archive[name] for name in archive.files}

full_gene_columns = preprocessing["full_gene_columns"].astype(str).tolist()
core_determinant_columns = preprocessing["core_determinant_columns"].astype(str).tolist()
point_target_columns = preprocessing["point_target_columns"].astype(str).tolist()
burden_columns = preprocessing["burden_columns"].astype(str).tolist()
full_gene_idf_weights = preprocessing["full_gene_idf_weights"].astype(np.float64)
core_determinant_idf_weights = preprocessing["core_determinant_idf_weights"].astype(np.float64)
point_target_idf_weights = preprocessing["point_target_idf_weights"].astype(np.float64)
burden_means = preprocessing["burden_means"].astype(np.float64)
burden_standard_deviations = preprocessing["burden_standard_deviations"].astype(np.float64)
burden_gamma = float(preprocessing["burden_gamma"].reshape(-1)[0])

reference_full_normalized = sparse.load_npz(
    files15b["15B_pathogen_full_gene_representation.npz"]
).tocsr()
reference_core_normalized = sparse.load_npz(
    files15b["15B_pathogen_core_determinant_representation.npz"]
).tocsr()
reference_point_normalized = sparse.load_npz(
    files15b["15B_pathogen_point_target_representation.npz"]
).tocsr()
with np.load(files15b["15B_pathogen_burden_representation.npz"]) as archive:
    reference_burden_standardized = archive["standardized_burden_matrix"].astype(np.float64)

pathogen15b_index = (
    pd.read_csv(files15b["15B_pathogen_kernel_index.csv"])
    .sort_values("kernel_row")
    .reset_index(drop=True)
)

final_model_c = joblib.load(files26["26_final_model_c_reference_model.joblib"])
with np.load(files26["26_final_model_c_embeddings.npz"]) as archive:
    pathogen_embedding = archive["pathogen_embedding"].astype(np.float64)
    pathogen_eigenvalues = archive["pathogen_eigenvalues"].astype(np.float64)
    antibiotic_embedding = archive["antibiotic_embedding"].astype(np.float64)

model_c_pathogen_index = (
    pd.read_csv(files26["26_final_model_c_pathogen_embedding_index.csv"])
    .sort_values("pathogen_embedding_row")
    .reset_index(drop=True)
)
model_c_antibiotic_index = (
    pd.read_csv(files26["26_final_model_c_antibiotic_embedding_index.csv"])
    .sort_values("antibiotic_embedding_row")
    .reset_index(drop=True)
)
model_c_interactions = pd.read_csv(files26["26_model_c_aligned_interactions.csv.gz"])

with open(files26["26_final_model_c_configuration.json"], "r", encoding="utf-8") as input_file:
    model_c_configuration = json.load(input_file)
with open(files27["27_model_c_prediction_support_configuration.json"], "r", encoding="utf-8") as input_file:
    support_configuration = json.load(input_file)

prediction_interval_calibration = pd.read_csv(
    files28["28_model_c_antibiotic_prediction_interval_calibration.csv"]
)

locus_panel = pd.read_csv(notebook23_required_files["locus_panel"], dtype=str)
reference_audit = pd.read_csv(notebook23_required_files["reference_audit"], dtype=str)
alignment_validation = pd.read_csv(notebook24_required_files["alignment_validation"])
training_pathogen_order = pd.read_csv(notebook24_required_files["pathogen_order"])
training_available_locus_counts = np.load(notebook24_required_files["available_locus_counts"])

with open(notebook24_required_files["locus_kernel_metadata"], "r", encoding="utf-8") as input_file:
    locus_kernel_metadata = json.load(input_file)

assert len(full_gene_columns) == EXPECTED_FULL_GENE_FEATURES
assert len(core_determinant_columns) == EXPECTED_CORE_DETERMINANT_FEATURES
assert len(point_target_columns) == EXPECTED_POINT_TARGET_FEATURES
assert len(burden_columns) == EXPECTED_BURDEN_FEATURES
assert reference_full_normalized.shape == (EXPECTED_MODEL3B_PATHOGENS, EXPECTED_FULL_GENE_FEATURES)
assert reference_core_normalized.shape == (EXPECTED_MODEL3B_PATHOGENS, EXPECTED_CORE_DETERMINANT_FEATURES)
assert reference_point_normalized.shape == (EXPECTED_MODEL3B_PATHOGENS, EXPECTED_POINT_TARGET_FEATURES)
assert reference_burden_standardized.shape == (EXPECTED_MODEL3B_PATHOGENS, EXPECTED_BURDEN_FEATURES)
assert pathogen_embedding.shape == (EXPECTED_MODEL_C_PATHOGENS, EXPECTED_PATHOGEN_COORDINATES)
assert pathogen_eigenvalues.shape == (EXPECTED_PATHOGEN_COORDINATES,)
assert antibiotic_embedding.shape == (EXPECTED_ANTIBIOTICS, EXPECTED_ANTIBIOTIC_COORDINATES)
assert np.asarray(final_model_c.coef_).shape == (EXPECTED_INTERACTION_FEATURES,)
assert len(locus_panel) == EXPECTED_TARGETED_LOCI
assert len(reference_audit) == EXPECTED_TARGETED_LOCI
assert training_available_locus_counts.shape == (EXPECTED_MODEL_C_PATHOGENS,)
assert len(prediction_interval_calibration) == EXPECTED_ANTIBIOTICS
assert set(prediction_interval_calibration["antibiotic"].astype(str)) == set(
    model_c_antibiotic_index["antibiotic"].astype(str)
)

configuration_checks = {
    "sequence_kernel": model_c_configuration["selected_sequence_kernel"] == SELECTED_SEQUENCE_KERNEL,
    "rho": np.isclose(model_c_configuration["selected_rho"], SELECTED_RHO),
    "pathogen_coordinates": model_c_configuration["selected_pathogen_dimension"] == EXPECTED_PATHOGEN_COORDINATES,
    "ridge_alpha": np.isclose(model_c_configuration["selected_ridge_alpha"], SELECTED_RIDGE_ALPHA),
    "interaction_features": model_c_configuration["interaction_predictors"] == EXPECTED_INTERACTION_FEATURES,
    "locus_kernel_normalisation": "geometric mean" in locus_kernel_metadata["normalisation"],
}
if not all(configuration_checks.values()):
    raise ValueError(f"Fixed Model C configuration mismatch: {configuration_checks}")

model3b_rows = model_c_pathogen_index["model_3b_row_index"].to_numpy(dtype=int)
if not np.array_equal(
    pathogen15b_index.iloc[model3b_rows]["biosample"].astype(str).to_numpy(),
    model_c_pathogen_index["biosample"].astype(str).to_numpy(),
):
    raise ValueError("The Model 3B and Model C pathogen rows do not match.")

if not np.array_equal(
    training_pathogen_order["biosample"].astype(str).to_numpy(),
    model_c_pathogen_index["biosample"].astype(str).to_numpy(),
):
    raise ValueError("The Notebook 24 and Notebook 26 pathogen orders do not match.")

represented_alignment_table = alignment_validation[
    alignment_validation["validation_status"] == "passed"
].copy()
if len(represented_alignment_table) != EXPECTED_REPRESENTED_LOCI:
    raise ValueError(
        f"Expected {EXPECTED_REPRESENTED_LOCI} represented loci; observed {len(represented_alignment_table)}."
    )

missing_alignment_files = []
for locus_id in represented_alignment_table["locus_id"]:
    for suffix in ["alignment.fasta.gz", "alignment.json"]:
        path = NOTEBOOK24_ALIGNMENT_DIRECTORY / f"24_{locus_id}_{suffix}"
        if not path.exists():
            missing_alignment_files.append(path)
if missing_alignment_files:
    raise FileNotFoundError(f"Notebook 24 alignment files are missing: {missing_alignment_files[:10]}")

fixed_input_validation = pd.DataFrame([
    {"component": "Model 3B pathogen order", "rows": len(pathogen15b_index), "status": "passed"},
    {"component": "Model C pathogen order", "rows": len(model_c_pathogen_index), "status": "passed"},
    {"component": "Targeted locus panel", "rows": len(locus_panel), "status": "passed"},
    {"component": "Represented locus alignments", "rows": len(represented_alignment_table), "status": "passed"},
    {"component": "Prediction-interval calibration", "rows": len(prediction_interval_calibration), "status": "passed"},
])
display(fixed_input_validation)

print("All fixed reference representations and their row orders passed validation.")
print("\nTransition: Cell 30.5 will accept one new E. coli BioSample and identify its assembly accession in the fixed NCBI metadata release.")


In [ ]:
#@title Cell 30.5 - Select a new E. coli BioSample and assembly
# This cell finds the fixed NCBI AMR metadata release, retrieves one BioSample record and identifies one unambiguous assembly accession.

if NCBI_METADATA_PATH_OVERRIDE.strip():
    NCBI_METADATA_PATH = Path(NCBI_METADATA_PATH_OVERRIDE.strip())
    if not NCBI_METADATA_PATH.exists():
        raise FileNotFoundError(f"The specified metadata file does not exist: {NCBI_METADATA_PATH}")
else:
    metadata_candidates = sorted(
        path for path in Path("/content/drive/MyDrive").rglob("*.amr.metadata.tsv")
        if path.is_file()
    )
    if len(metadata_candidates) == 0:
        raise FileNotFoundError("No file ending in '.amr.metadata.tsv' was found in MyDrive.")
    if len(metadata_candidates) > 1:
        display(pd.DataFrame({"metadata_path": [str(path) for path in metadata_candidates]}))
        raise ValueError(
            "More than one fixed NCBI metadata release was found. Set NCBI_METADATA_PATH_OVERRIDE in Cell 30.2."
        )
    NCBI_METADATA_PATH = metadata_candidates[0]

QUERY_BIOSAMPLE = input(
    "Enter the new E. coli BioSample accession (for example, SAMN02911890): "
).strip().upper()
if not re.fullmatch(r"SAMN\d+", QUERY_BIOSAMPLE):
    raise ValueError("The BioSample accession must begin with SAMN and contain digits only after SAMN.")

if QUERY_BIOSAMPLE in set(model_c_pathogen_index["biosample"].astype(str)):
    raise ValueError(
        "This BioSample is already one of the 9,058 Model C training pathogens. "
        "Notebook 30 is reserved for a new pathogen; use Notebook 28 for a training-cohort pathogen."
    )

metadata_header = pd.read_csv(NCBI_METADATA_PATH, sep="\t", nrows=0)
biosample_candidates = [
    column for column in ["biosample_acc", "biosample", "BioSample", "biosample_accession"]
    if column in metadata_header.columns
]
if len(biosample_candidates) != 1:
    raise ValueError(f"Could not identify one BioSample column: {metadata_header.columns.tolist()}")
BIOSAMPLE_COLUMN = biosample_candidates[0]

matching_chunks = []
for metadata_chunk in pd.read_csv(
    NCBI_METADATA_PATH,
    sep="\t",
    dtype=str,
    chunksize=250_000,
    keep_default_na=False,
    low_memory=False,
):
    matches = metadata_chunk.loc[
        metadata_chunk[BIOSAMPLE_COLUMN].astype(str).str.upper() == QUERY_BIOSAMPLE
    ].copy()
    if not matches.empty:
        matching_chunks.append(matches)

if not matching_chunks:
    raise LookupError(f"{QUERY_BIOSAMPLE} was not found in the fixed NCBI AMR metadata release.")

query_amr_records = pd.concat(matching_chunks, ignore_index=True)
if "scientific_name" not in query_amr_records.columns:
    raise ValueError("The fixed metadata release does not contain scientific_name.")
if not query_amr_records["scientific_name"].astype(str).str.contains(
    r"^Escherichia coli(?:\s|$)", case=False, regex=True
).all():
    raise ValueError("The selected BioSample is not consistently identified as Escherichia coli.")

assembly_columns = [
    column
    for column in query_amr_records.columns
    if (
        "assembly" in column.lower()
        or "asm_acc" in column.lower()
    )
]
if not assembly_columns:
    raise ValueError("No assembly-accession column was found in the fixed metadata release.")

assembly_values = []
for column in assembly_columns:
    assembly_values.extend(query_amr_records[column].astype(str).tolist())
assembly_accessions = sorted({
    match.group(0)
    for value in assembly_values
    for match in [re.search(r"GC[AF]_\d+\.\d+", value.upper())]
    if match is not None
})

if len(assembly_accessions) == 0:
    raise ValueError(f"No versioned GCA or GCF assembly accession was found for {QUERY_BIOSAMPLE}.")
if len(assembly_accessions) == 1:
    QUERY_ASSEMBLY_ACCESSION = assembly_accessions[0]
else:
    display(pd.DataFrame({"candidate_assembly_accession": assembly_accessions}))
    QUERY_ASSEMBLY_ACCESSION = input(
        "Enter exactly one displayed assembly accession for this BioSample: "
    ).strip().upper()
    if QUERY_ASSEMBLY_ACCESSION not in assembly_accessions:
        raise ValueError("The selected assembly accession is not one of the displayed candidates.")

assembly_row_mask = np.zeros(len(query_amr_records), dtype=bool)
for column in assembly_columns:
    assembly_row_mask |= query_amr_records[column].astype(str).str.contains(
        re.escape(QUERY_ASSEMBLY_ACCESSION), case=False, regex=True
    ).to_numpy()
selected_query_records = query_amr_records.loc[assembly_row_mask].copy()
if selected_query_records.empty:
    raise ValueError("The selected assembly could not be mapped back to its metadata record.")

biological_fields = [
    column for column in [
        "AMR_genotypes", "AMR_genotypes_core", "number_amr_genes", "number_core_amr_genes"
    ] if column in selected_query_records.columns
]
required_biological_fields = {
    "AMR_genotypes",
    "AMR_genotypes_core",
    "number_amr_genes",
    "number_core_amr_genes",
}
if set(biological_fields) != required_biological_fields:
    raise ValueError(
        "The fixed NCBI metadata record is missing required Model 3B fields: "
        f"{sorted(required_biological_fields - set(biological_fields))}"
    )
if selected_query_records[biological_fields].drop_duplicates().shape[0] != 1:
    display(selected_query_records[[BIOSAMPLE_COLUMN, *assembly_columns, *biological_fields]])
    raise ValueError("The selected assembly has conflicting AMR metadata records.")

query_record = selected_query_records.iloc[0]
QUERY_METADATA_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_ncbi_amr_record.csv"
selected_query_records.to_csv(QUERY_METADATA_PATH, index=False)

query_selection_summary = pd.DataFrame([{
    "biosample": QUERY_BIOSAMPLE,
    "assembly_accession": QUERY_ASSEMBLY_ACCESSION,
    "scientific_name": query_record["scientific_name"],
    "metadata_records": len(selected_query_records),
    "reference_cohort_member": False,
}])
display(query_selection_summary)

print(f"Saved: {QUERY_METADATA_PATH}")
print("\nTransition: Cell 30.6 will convert the fixed NCBI fields into the exact Model 3B pathogen features.")


In [ ]:
#@title Cell 30.6 - Construct the exact Model 3B query features
# This cell constructs the 267 AMR-genotype, 833 core-genotype, 25 target-locus and five numerical features used by Model 3B.

def parse_ncbi_list(value):
    text = str(value).strip()
    if not text or text.upper() in {"NULL", "NA", "NAN", "NONE"}:
        return []
    return [
        item.strip() for item in text.split(",")
        if item.strip() and item.strip().upper() not in {"NULL", "NA", "NAN", "NONE"}
    ]

reported_full_genes = parse_ncbi_list(query_record["AMR_genotypes"])
reported_core_determinants = parse_ncbi_list(query_record["AMR_genotypes_core"])

full_gene_lookup = {
    column.removeprefix("full_gene__").casefold(): index
    for index, column in enumerate(full_gene_columns)
}
core_determinant_lookup = {
    column.removeprefix("determinant__").casefold(): index
    for index, column in enumerate(core_determinant_columns)
}
point_target_lookup = {
    column.removeprefix("point_target__").casefold(): index
    for index, column in enumerate(point_target_columns)
}

query_full_raw = np.zeros(EXPECTED_FULL_GENE_FEATURES, dtype=np.float64)
query_core_raw = np.zeros(EXPECTED_CORE_DETERMINANT_FEATURES, dtype=np.float64)
query_point_raw = np.zeros(EXPECTED_POINT_TARGET_FEATURES, dtype=np.float64)

recognised_full_genes = []
unmatched_full_genes = []
for gene in reported_full_genes:
    feature_index = full_gene_lookup.get(gene.casefold())
    if feature_index is None:
        unmatched_full_genes.append(gene)
    else:
        query_full_raw[feature_index] = 1.0
        recognised_full_genes.append(gene)

recognised_core_determinants = []
unmatched_core_determinants = []
for determinant in reported_core_determinants:
    feature_index = core_determinant_lookup.get(determinant.casefold())
    if feature_index is None:
        unmatched_core_determinants.append(determinant)
    else:
        query_core_raw[feature_index] = 1.0
        recognised_core_determinants.append(determinant)

recognised_point_targets = set()
for determinant in reported_core_determinants:
    determinant_lower = determinant.casefold()
    for target_name, feature_index in point_target_lookup.items():
        if determinant_lower == target_name or re.match(
            rf"^{re.escape(target_name)}(?:[^a-z0-9]|$)", determinant_lower
        ):
            query_point_raw[feature_index] = 1.0
            recognised_point_targets.add(target_name)
            break

partial_core_calls = [
    determinant for determinant in reported_core_determinants
    if "partial" in determinant.casefold()
]
mutation_core_calls = [
    determinant for determinant in reported_core_determinants
    if any(
        re.match(rf"^{re.escape(target)}(?:[^a-z0-9]|$)", determinant.casefold())
        for target in point_target_lookup
    )
]
core_gene_calls = [
    determinant for determinant in reported_core_determinants
    if determinant not in mutation_core_calls and determinant not in partial_core_calls
]

query_burden_raw = np.array([
    len(reported_full_genes),
    len(reported_core_determinants),
    len(core_gene_calls),
    len(mutation_core_calls),
    len(partial_core_calls),
], dtype=np.float64)

if int(float(query_record["number_amr_genes"])) != len(reported_full_genes):
    raise ValueError("number_amr_genes does not match the parsed AMR_genotypes field.")
if int(float(query_record["number_core_amr_genes"])) != len(reported_core_determinants):
    raise ValueError("number_core_amr_genes does not match the parsed AMR_genotypes_core field.")

query_feature_summary = pd.DataFrame([
    {
        "component": "1",
        "name": "AMR_genotypes",
        "definition": "267 binary features taken directly from AMR_genotypes",
        "reported": len(reported_full_genes),
        "recognised": int(query_full_raw.sum()),
        "unmatched": len(unmatched_full_genes),
        "example_values": ",".join(reported_full_genes[:5]),
    },
    {
        "component": "2",
        "name": "AMR_genotypes_core",
        "definition": "833 binary features taken directly from AMR_genotypes_core",
        "reported": len(reported_core_determinants),
        "recognised": int(query_core_raw.sum()),
        "unmatched": len(unmatched_core_determinants),
        "example_values": ",".join(reported_core_determinants[:5]),
    },
    {
        "component": "3",
        "name": "Target-locus features",
        "definition": "25 binary target-locus features derived from Component 2",
        "reported": len(mutation_core_calls),
        "recognised": int(query_point_raw.sum()),
        "unmatched": len(mutation_core_calls) - int(query_point_raw.sum()),
        "example_values": ",".join(sorted(recognised_point_targets)[:5]),
    },
    {
        "component": "4",
        "name": "AMR counts",
        "definition": "Five numerical counts calculated from AMR_genotypes and AMR_genotypes_core",
        "reported": 5,
        "recognised": 5,
        "unmatched": 0,
        "example_values": ",".join(f"{name}={value:g}" for name, value in zip(burden_columns, query_burden_raw)),
    },
])

QUERY_FEATURE_SUMMARY_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_model3b_feature_summary.csv"
UNMATCHED_FULL_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_unmatched_full_genes.csv"
UNMATCHED_CORE_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_unmatched_core_determinants.csv"
QUERY_FEATURE_VECTOR_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_model3b_feature_vectors.npz"

query_feature_summary.to_csv(QUERY_FEATURE_SUMMARY_PATH, index=False)
pd.DataFrame({"unmatched_full_gene": unmatched_full_genes}).to_csv(UNMATCHED_FULL_PATH, index=False)
pd.DataFrame({"unmatched_core_determinant": unmatched_core_determinants}).to_csv(UNMATCHED_CORE_PATH, index=False)
np.savez_compressed(
    QUERY_FEATURE_VECTOR_PATH,
    full_gene=query_full_raw,
    core_determinant=query_core_raw,
    point_target=query_point_raw,
    burden=query_burden_raw,
)

display(query_feature_summary)
print("The exact Model 3B query features were constructed and saved.")
print("\nTransition: Cell 30.7 will calculate the query's Model 3B similarity to the 9,058 ordered Model C training pathogens.")


In [ ]:
#@title Cell 30.7 - Calculate Model 3B similarity to the training pathogens
# This cell applies the fixed Model 3B feature weights and calculates one Model 3B similarity value for each ordered Model C training pathogen.

def normalise_query_vector(raw_vector, weights):
    weighted_vector = raw_vector * weights
    vector_length = np.linalg.norm(weighted_vector)
    if vector_length == 0:
        return np.zeros_like(weighted_vector)
    return weighted_vector / vector_length

query_full_normalized = normalise_query_vector(query_full_raw, full_gene_idf_weights)
query_core_normalized = normalise_query_vector(query_core_raw, core_determinant_idf_weights)
query_point_normalized = normalise_query_vector(query_point_raw, point_target_idf_weights)
query_burden_standardized = (query_burden_raw - burden_means) / burden_standard_deviations

full_gene_similarity_9377 = np.asarray(
    reference_full_normalized @ query_full_normalized
).reshape(-1)
core_similarity_9377 = np.asarray(
    reference_core_normalized @ query_core_normalized
).reshape(-1)
point_similarity_9377 = np.asarray(
    reference_point_normalized @ query_point_normalized
).reshape(-1)
burden_squared_distances = np.sum(
    (reference_burden_standardized - query_burden_standardized) ** 2,
    axis=1,
)
burden_similarity_9377 = np.exp(-burden_gamma * burden_squared_distances)

model3b_similarity_9377 = (
    FULL_GENE_WEIGHT * full_gene_similarity_9377
    + CORE_DETERMINANT_WEIGHT * core_similarity_9377
    + POINT_TARGET_WEIGHT * point_similarity_9377
    + BURDEN_WEIGHT * burden_similarity_9377
)

model3b_similarity = model3b_similarity_9377[model3b_rows]
full_gene_similarity = full_gene_similarity_9377[model3b_rows]
core_similarity = core_similarity_9377[model3b_rows]
point_similarity = point_similarity_9377[model3b_rows]
burden_similarity = burden_similarity_9377[model3b_rows]

if model3b_similarity.shape != (EXPECTED_MODEL_C_PATHOGENS,):
    raise ValueError(f"Unexpected Model 3B similarity dimension: {model3b_similarity.shape}")
if not np.isfinite(model3b_similarity).all():
    raise ValueError("The Model 3B similarity vector contains invalid values.")

model3b_similarity_table = model_c_pathogen_index[
    ["model_c_row_index", "model_3b_row_index", "biosample", "assembly_accession"]
].copy()
model3b_similarity_table["full_gene_similarity"] = full_gene_similarity
model3b_similarity_table["core_determinant_similarity"] = core_similarity
model3b_similarity_table["target_locus_similarity"] = point_similarity
model3b_similarity_table["burden_similarity"] = burden_similarity
model3b_similarity_table["model3b_pathogen_similarity"] = model3b_similarity

MODEL3B_SIMILARITY_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_model3b_similarity_to_training_pathogens.csv.gz"
model3b_similarity_table.to_csv(MODEL3B_SIMILARITY_PATH, index=False, compression="gzip")

display(
    model3b_similarity_table.sort_values(
        "model3b_pathogen_similarity", ascending=False
    ).head(10)
)
print(f"Saved: {MODEL3B_SIMILARITY_PATH}")
print("\nTransition: Cell 30.8 will install and validate the exact sequence-extraction and alignment software.")


In [ ]:
#@title Cell 30.8 - Install and validate sequence software
# This cell installs the same NCBI Datasets, AMRFinderPlus, BLASTN and MAFFT tools used by Notebooks 23 and 24 and records their versions.

if platform.machine().lower() not in {"x86_64", "amd64"}:
    raise RuntimeError(f"Unsupported Colab processor: {platform.machine()}")

TOOLS_DIRECTORY = WORK_DIRECTORY / "bin"
TOOLS_DIRECTORY.mkdir(parents=True, exist_ok=True)
datasets_executable = TOOLS_DIRECTORY / "datasets"

if not datasets_executable.exists():
    subprocess.run([
        "curl", "--fail", "--location", "--output", str(datasets_executable),
        "https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets",
    ], check=True)
datasets_executable.chmod(0o755)


def ensure_amrfinderplus():
    environment_directory = WORK_DIRECTORY / "amrfinder_environment"
    amrfinder_path = environment_directory / "bin" / "amrfinder"
    if not amrfinder_path.exists():
        print(f"Installing AMRFinderPlus {EXPECTED_AMRFINDERPLUS_VERSION}...")
        micromamba_path = TOOLS_DIRECTORY / "micromamba"
        if not micromamba_path.exists():
            micromamba_archive_path = WORK_DIRECTORY / "micromamba.tar.bz2"
            with urllib.request.urlopen(
                "https://micro.mamba.pm/api/micromamba/linux-64/latest", timeout=180
            ) as response, open(micromamba_archive_path, "wb") as output_file:
                shutil.copyfileobj(response, output_file)
            with tarfile.open(micromamba_archive_path, "r:bz2") as archive:
                extracted = archive.extractfile(archive.getmember("bin/micromamba"))
                if extracted is None:
                    raise RuntimeError("The micromamba executable could not be extracted.")
                with open(micromamba_path, "wb") as output_file:
                    shutil.copyfileobj(extracted, output_file)
            micromamba_path.chmod(0o755)
            micromamba_archive_path.unlink(missing_ok=True)

        mamba_environment = os.environ.copy()
        mamba_environment["MAMBA_ROOT_PREFIX"] = str(WORK_DIRECTORY / "micromamba_root")
        subprocess.run([
            str(micromamba_path), "create", "--yes",
            "--prefix", str(environment_directory),
            "--override-channels", "--channel-priority", "strict",
            "--channel", "conda-forge", "--channel", "bioconda",
            f"ncbi-amrfinderplus={EXPECTED_AMRFINDERPLUS_VERSION}",
        ], check=True, env=mamba_environment)

    software_environment = os.environ.copy()
    software_environment["PATH"] = (
        str(amrfinder_path.parent) + os.pathsep + software_environment.get("PATH", "")
    )
    database_check = subprocess.run(
        [str(amrfinder_path), "--database_version"],
        capture_output=True, text=True, env=software_environment,
    )
    if database_check.returncode != 0:
        print("Downloading the AMRFinderPlus database...")
        subprocess.run([str(amrfinder_path), "-u"], check=True, env=software_environment)
    return amrfinder_path, software_environment


amrfinder_executable, command_environment = ensure_amrfinderplus()
blastn_executable = amrfinder_executable.parent / "blastn"
if not blastn_executable.exists():
    raise FileNotFoundError("BLASTN was not found in the AMRFinderPlus environment.")

MAFFT_EXECUTABLE = shutil.which("mafft")
if MAFFT_EXECUTABLE is None:
    print("Installing MAFFT...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "mafft"], check=True)
    MAFFT_EXECUTABLE = shutil.which("mafft")
if MAFFT_EXECUTABLE is None:
    raise FileNotFoundError("MAFFT was not installed successfully.")

datasets_version = subprocess.run(
    [str(datasets_executable), "--version"], capture_output=True, text=True, check=True
).stdout.strip()
amrfinder_version_result = subprocess.run(
    [str(amrfinder_executable), "--version"],
    capture_output=True, text=True, check=True, env=command_environment,
)
amrfinder_version = amrfinder_version_result.stdout.strip() or amrfinder_version_result.stderr.strip()
database_version_result = subprocess.run(
    [str(amrfinder_executable), "--database_version"],
    capture_output=True, text=True, check=True, env=command_environment,
)
amrfinder_database_version = (
    database_version_result.stdout.strip() or database_version_result.stderr.strip()
)
blastn_version = subprocess.run(
    [str(blastn_executable), "-version"],
    capture_output=True, text=True, check=True, env=command_environment,
).stdout.splitlines()[0]
mafft_result = subprocess.run(
    [MAFFT_EXECUTABLE, "--version"], capture_output=True, text=True, check=False
)
mafft_version = mafft_result.stderr.strip() or mafft_result.stdout.strip()

if EXPECTED_AMRFINDERPLUS_VERSION not in amrfinder_version:
    raise ValueError(f"Expected AMRFinderPlus {EXPECTED_AMRFINDERPLUS_VERSION}; observed {amrfinder_version}")
if EXPECTED_AMRFINDER_DATABASE_VERSION not in amrfinder_database_version:
    raise ValueError(
        "The AMRFinderPlus database does not match Notebook 23. "
        f"Expected {EXPECTED_AMRFINDER_DATABASE_VERSION}; observed {amrfinder_database_version}."
    )

connection_test = subprocess.run(
    [
        str(datasets_executable), "summary", "genome", "accession",
        "GCF_000005845.2", "--as-json-lines",
    ],
    capture_output=True,
    text=True,
    timeout=120,
    env=command_environment,
)
if connection_test.returncode != 0 or not connection_test.stdout.strip():
    raise RuntimeError(
        "The NCBI Datasets connection test failed: "
        + (connection_test.stderr.strip() or "no metadata returned")
    )
json.loads(connection_test.stdout.splitlines()[0])

software_versions = pd.DataFrame([
    {"software": "NCBI Datasets", "version": datasets_version},
    {"software": "AMRFinderPlus", "version": amrfinder_version},
    {"software": "AMRFinderPlus database", "version": amrfinder_database_version},
    {"software": "BLASTN", "version": blastn_version},
    {"software": "MAFFT", "version": mafft_version},
])
SOFTWARE_VERSION_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_software_versions.csv"
software_versions.to_csv(SOFTWARE_VERSION_PATH, index=False)
display(software_versions)

print("All required sequence programs and the fixed AMRFinderPlus database passed validation.")
print("\nTransition: Cell 30.9 will extract the query's targeted AMR-related sequences using the final Notebook 23 rules.")


In [ ]:
#@title Cell 30.9 - Extract the query's targeted AMR-related sequences
# This cell applies the final Notebook 23 AMRFinderPlus, blaTEM-upstream and MG1655-reference BLASTN rules to one query assembly.

full_gene_rule = "pathogen-specific AMRFinderPlus nucleotide sequence"
coding_reference_rule = "E. coli K-12 MG1655 reference gene"

full_gene_panel = reference_audit[
    reference_audit["reference_rule"] == full_gene_rule
].copy()
coding_reference_panel = reference_audit[
    reference_audit["reference_rule"] == coding_reference_rule
].copy()
bla_temp_panel = reference_audit[
    reference_audit["locus_name"] == "blaTEMp"
].copy()

assert len(full_gene_panel) == EXPECTED_FULL_GENE_LOCI
assert len(coding_reference_panel) == EXPECTED_CODING_REFERENCE_LOCI
assert len(bla_temp_panel) == 1

if coding_reference_panel["reference_sequence"].isna().any() or (
    coding_reference_panel["reference_sequence"] == ""
).any():
    raise ValueError("One or more fixed MG1655 coding-reference sequences are missing.")

query_fasta_path = WORK_DIRECTORY / "30_coding_reference_queries.fasta"
with open(query_fasta_path, "w", encoding="utf-8") as fasta_file:
    for reference in coding_reference_panel.itertuples(index=False):
        sequence = str(reference.reference_sequence).upper()
        fasta_file.write(f">{reference.locus_id}\n")
        for start in range(0, len(sequence), 80):
            fasta_file.write(sequence[start:start + 80] + "\n")

STOP_CODONS = {"TAA", "TAG", "TGA"}
BACTERIAL_START_CODONS = {"ATG", "GTG", "TTG", "CTG", "ATT", "ATC", "ATA"}
IUPAC_NUCLEOTIDES = set("ACGTRYSWKMBDHVN")

REVIEWED_CODING_HITS = {
    ("GCA_000522325.1", "LOCUS_0281"): {
        "expected_contig": "KI929782.1",
        "minimum_identity": 89.0,
        "minimum_coverage": 99.0,
        "review_reason": "reviewed full-length ompC hit; distinct from ompF",
    }
}

BLAST_COLUMNS = [
    "locus_id", "contig_id", "percent_identity", "alignment_length",
    "reference_length", "query_start", "query_stop", "subject_start",
    "subject_stop", "bit_score", "e_value",
]

full_gene_records = full_gene_panel.to_dict(orient="records")
full_gene_by_exact = {}
full_gene_by_base = {}
for record in full_gene_records:
    exact_key = str(record["locus_name"]).strip().casefold()
    base_key = str(record["locus_name"]).split("=", 1)[0].strip().casefold()
    full_gene_by_exact.setdefault(exact_key, []).append(record)
    full_gene_by_base.setdefault(base_key, []).append(record)


def reverse_complement(sequence):
    table = str.maketrans(
        "ACGTRYKMSWBDHVNacgtrykmswbdhvn",
        "TGCAYRMKSWVHDBNtgcayrmkswvhdbn",
    )
    return sequence.translate(table)[::-1]


def assess_coding_integrity(
    sequence,
    reference_sequence,
    touches_contig_edge,
):
    sequence = str(sequence).upper()
    reference_sequence = str(reference_sequence).upper()

    ambiguous_nucleotides = sum(
        nucleotide not in {"A", "C", "G", "T"}
        for nucleotide in sequence
    )
    length_difference = (
        len(sequence) - len(reference_sequence)
    )
    length_is_multiple_of_three = len(sequence) % 3 == 0

    complete_codons = [
        sequence[position:position + 3]
        for position in range(0, len(sequence) - 2, 3)
    ]
    start_codon = (
        complete_codons[0] if complete_codons else ""
    )
    terminal_codon = (
        complete_codons[-1] if complete_codons else ""
    )
    internal_stop_count = sum(
        codon in STOP_CODONS
        for codon in complete_codons[:-1]
    )

    reference_has_terminal_stop = (
        len(reference_sequence) >= 3
        and reference_sequence[-3:] in STOP_CODONS
    )
    recognised_start = (
        start_codon in BACTERIAL_START_CODONS
    )
    terminal_stop_present = (
        terminal_codon in STOP_CODONS
    )

    uncertainty_reasons = []
    disruption_reasons = []

    if touches_contig_edge:
        uncertainty_reasons.append(
            "sequence reaches a contig boundary"
        )
    if ambiguous_nucleotides > 0:
        uncertainty_reasons.append(
            f"{ambiguous_nucleotides} ambiguous nucleotide(s)"
        )
    if not length_is_multiple_of_three:
        disruption_reasons.append(
            "length is not a multiple of three"
        )
    if internal_stop_count > 0:
        disruption_reasons.append(
            f"{internal_stop_count} internal stop codon(s)"
        )
    if not recognised_start:
        disruption_reasons.append(
            f"unrecognised start codon {start_codon or 'none'}"
        )
    if (
        reference_has_terminal_stop
        and not terminal_stop_present
    ):
        disruption_reasons.append(
            "terminal stop codon is absent"
        )

    if uncertainty_reasons:
        coding_integrity = "uncertain"
        integrity_reason = "; ".join(
            uncertainty_reasons + disruption_reasons
        )
    elif disruption_reasons:
        coding_integrity = "potentially disrupted"
        integrity_reason = "; ".join(disruption_reasons)
    else:
        coding_integrity = "intact ORF"
        if length_difference == 0:
            integrity_reason = "no coding disruption detected"
        else:
            integrity_reason = (
                "in-frame length difference of "
                f"{length_difference:+d} nucleotide(s)"
            )

    return {
        "coding_integrity": coding_integrity,
        "integrity_reason": integrity_reason,
        "length_difference": length_difference,
        "ambiguous_nucleotides":
            ambiguous_nucleotides,
        "start_codon": start_codon,
        "internal_stop_count": internal_stop_count,
        "terminal_codon": terminal_codon,
    }


def clean_column_name(column_name):
    cleaned = re.sub(
        r"[^a-z0-9]+",
        "_",
        str(column_name).strip().lower(),
    ).strip("_")
    return cleaned


def find_column(table, aliases, required=True):
    normalized_lookup = {
        clean_column_name(column): column
        for column in table.columns
    }
    for alias in aliases:
        normalized_alias = clean_column_name(alias)
        if normalized_alias in normalized_lookup:
            return normalized_lookup[normalized_alias]

    if required:
        raise ValueError(
            "Required AMRFinderPlus column was not found. "
            f"Expected one of {aliases}; observed "
            f"{list(table.columns)}"
        )
    return None


def safe_float(value):
    try:
        if pd.isna(value):
            return np.nan
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def resolve_contig_id(reported_contig_id, assembly_sequences):
    reported = str(reported_contig_id).strip()
    if reported in assembly_sequences:
        return reported

    variants = {
        reported,
        reported.removeprefix("lcl|"),
        reported.split()[0],
    }
    for contig_id in assembly_sequences:
        contig_variants = {
            contig_id,
            contig_id.removeprefix("lcl|"),
            contig_id.split()[0],
        }
        if variants & contig_variants:
            return contig_id

    raise KeyError(
        f"Contig {reported_contig_id} was not found "
        "in the assembly FASTA."
    )


def extract_genomic_interval(
    assembly_sequences,
    contig_id,
    start,
    stop,
    strand,
):
    resolved_contig = resolve_contig_id(
        contig_id,
        assembly_sequences,
    )
    contig_sequence = assembly_sequences[resolved_contig]

    left = min(int(start), int(stop))
    right = max(int(start), int(stop))
    left = max(1, left)
    right = min(len(contig_sequence), right)

    sequence = contig_sequence[left - 1:right]
    normalized_strand = str(strand).strip()
    if normalized_strand == "-" or int(start) > int(stop):
        sequence = reverse_complement(sequence)
        normalized_strand = "-"
    else:
        normalized_strand = "+"

    return {
        "contig_id": resolved_contig,
        "contig_length": len(contig_sequence),
        "start": left,
        "stop": right,
        "strand": normalized_strand,
        "sequence": sequence,
        "sequence_length": len(sequence),
        "touches_contig_edge": (
            left == 1 or right == len(contig_sequence)
        ),
    }


def extract_reference_oriented_sequence(
    hit,
    assembly_sequences,
):
    contig_id = resolve_contig_id(
        hit["contig_id"],
        assembly_sequences,
    )
    contig_sequence = assembly_sequences[contig_id]

    query_start = int(hit["query_start"])
    query_stop = int(hit["query_stop"])
    reference_length = int(hit["reference_length"])
    subject_start = int(hit["subject_start"])
    subject_stop = int(hit["subject_stop"])

    if subject_start <= subject_stop:
        strand = "+"
        extraction_start = subject_start - (query_start - 1)
        extraction_stop = (
            subject_stop + (reference_length - query_stop)
        )
    else:
        strand = "-"
        extraction_start = (
            subject_stop
            - (reference_length - query_stop)
        )
        extraction_stop = (
            subject_start + (query_start - 1)
        )

    unclipped_start = extraction_start
    unclipped_stop = extraction_stop
    extraction_start = max(1, extraction_start)
    extraction_stop = min(
        len(contig_sequence),
        extraction_stop,
    )
    touches_contig_edge = (
        unclipped_start < 1
        or unclipped_stop > len(contig_sequence)
        or extraction_start == 1
        or extraction_stop == len(contig_sequence)
    )

    sequence = contig_sequence[
        extraction_start - 1:extraction_stop
    ]
    if strand == "-":
        sequence = reverse_complement(sequence)

    return {
        "contig_id": contig_id,
        "contig_length": len(contig_sequence),
        "start": extraction_start,
        "stop": extraction_stop,
        "strand": strand,
        "sequence": sequence,
        "sequence_length": len(sequence),
        "touches_contig_edge": touches_contig_edge,
    }


def label_base(label):
    return str(label).split("=", 1)[0].strip().casefold()


def choose_full_gene_locus(gene_symbol, report_row):
    symbol = str(gene_symbol).strip()
    exact_key = symbol.casefold()
    base_key = label_base(symbol)

    report_text = " ".join(
        str(value)
        for value in report_row.values
        if not pd.isna(value)
    ).upper()

    qualifier = None
    if (
        "MISTRANSLATION" in report_text
        or "FRAME_SHIFT" in report_text
        or "FRAMESHIFT" in report_text
    ):
        qualifier = "MISTRANSLATION"
    elif "PARTIAL" in report_text:
        qualifier = "PARTIAL"

    base_candidates = full_gene_by_base.get(
        base_key,
        [],
    )

    if qualifier is not None:
        qualified = [
            record
            for record in base_candidates
            if str(record["locus_name"]).upper().endswith(
                f"={qualifier}"
            )
        ]
        if len(qualified) == 1:
            return qualified[0], (
                f"matched {qualifier} AMRFinderPlus call"
            )

    exact_candidates = full_gene_by_exact.get(
        exact_key,
        [],
    )
    if len(exact_candidates) == 1:
        return exact_candidates[0], "exact gene-symbol match"

    unsuffixed = [
        record
        for record in base_candidates
        if "=" not in str(record["locus_name"])
    ]
    if len(unsuffixed) == 1:
        return unsuffixed[0], "base gene-symbol match"

    if len(base_candidates) == 1:
        return base_candidates[0], "unique base-label match"

    if len(base_candidates) == 0:
        return None, "AMR gene is not represented in the 266-locus panel"

    return None, (
        "AMR gene maps to multiple Model 3B labels "
        "and requires review"
    )


def run_amrfinder(
    biosample,
    assembly_accession,
    assembly_fasta_path,
    assembly_directory,
):
    report_path = assembly_directory / "amrfinder.tsv"
    command = [
        str(amrfinder_executable),
        "--nucleotide",
        str(assembly_fasta_path),
        "--organism",
        "Escherichia",
        "--plus",
        "--threads",
        str(THREADS),
        "--output",
        str(report_path),
    ]
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        env=command_environment,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"AMRFinderPlus failed for {assembly_accession}: "
            f"{result.stderr.strip()}"
        )

    if (
        not report_path.exists()
        or report_path.stat().st_size == 0
    ):
        return pd.DataFrame()

    report = pd.read_csv(
        report_path,
        sep="\t",
        dtype=str,
    )
    report.columns = [
        clean_column_name(column)
        for column in report.columns
    ]
    report.insert(0, "assembly_accession", assembly_accession)
    report.insert(0, "biosample", biosample)
    report.insert(
        2,
        "amrfinder_row_index",
        range(len(report)),
    )
    return report


def extract_amrfinder_sequences(
    biosample,
    assembly_accession,
    report,
    assembly_sequences,
):
    sequence_rows = []
    review_rows = []

    if report.empty:
        return sequence_rows, review_rows

    gene_column = find_column(
        report,
        [
            "gene_symbol",
            "genesymbol",
            "element_symbol",
            "elementsymbol",
        ],
    )
    contig_column = find_column(
        report,
        ["contig_id", "contig"],
    )
    start_column = find_column(report, ["start"])
    stop_column = find_column(report, ["stop"])
    strand_column = find_column(report, ["strand"])
    type_column = find_column(
        report,
        ["element_type", "type"],
        required=False,
    )
    subtype_column = find_column(
        report,
        ["element_subtype", "subtype"],
        required=False,
    )
    method_column = find_column(
        report,
        ["method"],
        required=False,
    )
    identity_column = find_column(
        report,
        [
            "identity_to_reference",
            "percent_identity_to_reference_sequence",
            "identity_to_reference_sequence",
        ],
        required=False,
    )
    coverage_column = find_column(
        report,
        [
            "coverage_of_reference",
            "percent_coverage_of_reference_sequence",
            "coverage_of_reference_sequence",
        ],
        required=False,
    )

    seen_gene_hits = set()
    seen_bla_tem_hits = set()

    for _, report_row in report.iterrows():
        gene_symbol = str(report_row[gene_column]).strip()
        if (
            gene_symbol == ""
            or gene_symbol.upper() == "NA"
            or pd.isna(report_row[gene_column])
        ):
            continue

        element_type = (
            str(report_row[type_column]).upper()
            if type_column is not None
            else ""
        )
        element_subtype = (
            str(report_row[subtype_column]).upper()
            if subtype_column is not None
            else ""
        )

        if type_column is not None and element_type != "AMR":
            continue
        if "POINT" in element_subtype:
            continue

        coordinate_values = [
            report_row[contig_column],
            report_row[start_column],
            report_row[stop_column],
            report_row[strand_column],
        ]
        if any(
            pd.isna(value) or str(value).strip() in {"", "NA"}
            for value in coordinate_values
        ):
            review_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession": assembly_accession,
                    "review_type":
                        "AMRFinderPlus coordinates missing",
                    "locus_id": "",
                    "locus_name": gene_symbol,
                    "details":
                        "Reported AMR hit has no extractable coordinates.",
                }
            )
            continue

        contig_id = resolve_contig_id(
            report_row[contig_column],
            assembly_sequences,
        )
        start = int(float(report_row[start_column]))
        stop = int(float(report_row[stop_column]))
        strand = str(report_row[strand_column]).strip()

        physical_key = (
            contig_id,
            min(start, stop),
            max(start, stop),
            strand,
            gene_symbol,
        )

        locus_record, mapping_reason = (
            choose_full_gene_locus(
                gene_symbol,
                report_row,
            )
        )

        if locus_record is not None:
            locus_key = (
                locus_record["locus_id"],
                physical_key,
            )
            if locus_key not in seen_gene_hits:
                seen_gene_hits.add(locus_key)
                extracted = extract_genomic_interval(
                    assembly_sequences,
                    contig_id,
                    start,
                    stop,
                    strand,
                )
                ambiguous_count = sum(
                    nucleotide not in {"A", "C", "G", "T"}
                    for nucleotide in extracted["sequence"]
                )
                sequence_status = (
                    "uncertain"
                    if (
                        extracted["touches_contig_edge"]
                        or ambiguous_count > 0
                    )
                    else "AMRFinderPlus sequence extracted"
                )
                sequence_rows.append(
                    {
                        "biosample": biosample,
                        "assembly_accession":
                            assembly_accession,
                        "locus_id":
                            locus_record["locus_id"],
                        "locus_name":
                            locus_record["locus_name"],
                        "locus_group":
                            locus_record["locus_group"],
                        "sequence_source":
                            "AMRFinderPlus nucleotide hit",
                        **extracted,
                        "accepted_for_kernel":
                            sequence_status
                            != "uncertain",
                        "sequence_status": sequence_status,
                        "coding_integrity":
                            "reported by AMRFinderPlus",
                        "integrity_reason": (
                            str(report_row[method_column])
                            if method_column is not None
                            else ""
                        ),
                        "length_difference": np.nan,
                        "ambiguous_nucleotides":
                            ambiguous_count,
                        "start_codon": "",
                        "internal_stop_count": np.nan,
                        "terminal_codon": "",
                        "percent_identity": (
                            safe_float(
                                report_row[identity_column]
                            )
                            if identity_column is not None
                            else np.nan
                        ),
                        "query_coverage": (
                            safe_float(
                                report_row[coverage_column]
                            )
                            if coverage_column is not None
                            else np.nan
                        ),
                        "acceptance_rule":
                            "AMRFinderPlus panel match",
                        "review_reason": mapping_reason,
                    }
                )

                if sequence_status == "uncertain":
                    review_rows.append(
                        {
                            "biosample": biosample,
                            "assembly_accession":
                                assembly_accession,
                            "review_type":
                                "uncertain AMR-gene sequence",
                            "locus_id":
                                locus_record["locus_id"],
                            "locus_name":
                                locus_record["locus_name"],
                            "details": (
                                "Sequence reaches a contig boundary "
                                "or contains ambiguous nucleotides."
                            ),
                        }
                    )
        else:
            review_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession": assembly_accession,
                    "review_type": "unmapped AMR gene",
                    "locus_id": "",
                    "locus_name": gene_symbol,
                    "details": mapping_reason,
                }
            )

        normalized_symbol = re.sub(
            r"[^a-z0-9]+",
            "",
            gene_symbol.casefold(),
        )
        if normalized_symbol.startswith("blatem"):
            bla_tem_key = (
                contig_id,
                min(start, stop),
                max(start, stop),
                strand,
            )
            if bla_tem_key in seen_bla_tem_hits:
                continue
            seen_bla_tem_hits.add(bla_tem_key)

            contig_sequence = assembly_sequences[contig_id]
            left = min(start, stop)
            right = max(start, stop)

            if strand == "-" or start > stop:
                promoter_start = right + 1
                promoter_stop = right + 100
                promoter_strand = "-"
            else:
                promoter_start = left - 100
                promoter_stop = left - 1
                promoter_strand = "+"

            if (
                promoter_start < 1
                or promoter_stop > len(contig_sequence)
            ):
                review_rows.append(
                    {
                        "biosample": biosample,
                        "assembly_accession":
                            assembly_accession,
                        "review_type":
                            "blaTEMp sequence incomplete",
                        "locus_id":
                            bla_temp_panel.iloc[0]["locus_id"],
                        "locus_name": "blaTEMp",
                        "details": (
                            "The 100-nucleotide upstream region "
                            "crosses a contig boundary."
                        ),
                    }
                )
                continue

            promoter = extract_genomic_interval(
                assembly_sequences,
                contig_id,
                promoter_start,
                promoter_stop,
                promoter_strand,
            )
            ambiguous_count = sum(
                nucleotide not in {"A", "C", "G", "T"}
                for nucleotide in promoter["sequence"]
            )
            sequence_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "locus_id":
                        bla_temp_panel.iloc[0]["locus_id"],
                    "locus_name": "blaTEMp",
                    "locus_group":
                        bla_temp_panel.iloc[0]["locus_group"],
                    "sequence_source":
                        "100 nucleotides immediately upstream "
                        "of an AMRFinderPlus blaTEM hit",
                    **promoter,
                    "accepted_for_kernel":
                        ambiguous_count == 0,
                    "sequence_status": (
                        "sequence extracted"
                        if ambiguous_count == 0
                        else "uncertain"
                    ),
                    "coding_integrity":
                        "not applicable: promoter",
                    "integrity_reason": "",
                    "length_difference": 0,
                    "ambiguous_nucleotides":
                        ambiguous_count,
                    "start_codon": "",
                    "internal_stop_count": np.nan,
                    "terminal_codon": "",
                    "percent_identity": np.nan,
                    "query_coverage": 100.0,
                    "acceptance_rule":
                        "blaTEM-oriented upstream extraction",
                    "review_reason": "",
                }
            )

    return sequence_rows, review_rows


def run_coding_reference_blast(
    biosample,
    assembly_accession,
    assembly_fasta_path,
    assembly_sequences,
    assembly_directory,
):
    blast_path = assembly_directory / "coding_blast.tsv"
    command = [
        str(blastn_executable),
        "-query",
        str(query_fasta_path),
        "-subject",
        str(assembly_fasta_path),
        "-task",
        "blastn",
        "-dust",
        "no",
        "-soft_masking",
        "false",
        "-evalue",
        "1e-20",
        "-max_target_seqs",
        "50",
        "-max_hsps",
        "1",
        "-num_threads",
        str(THREADS),
        "-outfmt",
        (
            "6 qseqid sseqid pident length qlen "
            "qstart qend sstart send bitscore evalue"
        ),
        "-out",
        str(blast_path),
    ]
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        env=command_environment,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"BLASTN failed for {assembly_accession}: "
            f"{result.stderr.strip()}"
        )

    if blast_path.stat().st_size == 0:
        blast_table = pd.DataFrame(columns=BLAST_COLUMNS)
    else:
        blast_table = pd.read_csv(
            blast_path,
            sep="\t",
            names=BLAST_COLUMNS,
        )

    for column in BLAST_COLUMNS[2:]:
        blast_table[column] = pd.to_numeric(
            blast_table[column],
            errors="coerce",
        )

    blast_table["aligned_reference_bases"] = (
        blast_table["query_stop"]
        - blast_table["query_start"]
    ).abs() + 1
    blast_table["query_coverage"] = (
        100.0
        * blast_table["aligned_reference_bases"]
        / blast_table["reference_length"]
    ).clip(upper=100.0)
    blast_table["passes_threshold"] = (
        (
            blast_table["percent_identity"]
            >= MINIMUM_NUCLEOTIDE_IDENTITY
        )
        & (
            blast_table["query_coverage"]
            >= MINIMUM_QUERY_COVERAGE
        )
    )

    sequence_rows = []
    best_hit_rows = []
    review_rows = []

    for locus in coding_reference_panel.itertuples(
        index=False
    ):
        locus_hits = blast_table[
            blast_table["locus_id"] == locus.locus_id
        ].sort_values(
            "bit_score",
            ascending=False,
        )

        accepted_hit = None
        acceptance_rule = ""
        review_reason = ""

        standard_hits = locus_hits[
            locus_hits["passes_threshold"]
        ]
        if not standard_hits.empty:
            accepted_hit = standard_hits.iloc[0]
            acceptance_rule = (
                "standard identity and coverage thresholds"
            )
        else:
            reviewed_key = (
                assembly_accession,
                locus.locus_id,
            )
            if reviewed_key in REVIEWED_CODING_HITS:
                if locus_hits.empty:
                    raise ValueError(
                        f"Reviewed hit {reviewed_key} is missing."
                    )
                reviewed_rule = REVIEWED_CODING_HITS[
                    reviewed_key
                ]
                reviewed_hit = locus_hits.iloc[0]
                if (
                    reviewed_hit["contig_id"]
                    != reviewed_rule["expected_contig"]
                    or reviewed_hit["percent_identity"]
                    < reviewed_rule["minimum_identity"]
                    or reviewed_hit["query_coverage"]
                    < reviewed_rule["minimum_coverage"]
                ):
                    raise ValueError(
                        f"Reviewed hit {reviewed_key} no longer "
                        "matches its reviewed evidence."
                    )
                accepted_hit = reviewed_hit
                acceptance_rule = "reviewed pilot decision"
                review_reason = reviewed_rule[
                    "review_reason"
                ]

            if (
                accepted_hit is None
                and locus.locus_name == "ompC"
                and not locus_hits.empty
            ):
                ompC_hit = locus_hits.iloc[0]
                if (
                    ompC_hit["percent_identity"]
                    >= OMPC_MINIMUM_NUCLEOTIDE_IDENTITY
                    and ompC_hit["query_coverage"]
                    >= OMPC_MINIMUM_QUERY_COVERAGE
                ):
                    accepted_hit = ompC_hit
                    acceptance_rule = (
                        "full-length ompC locus-specific thresholds"
                    )
                    review_reason = (
                        "full-length ompC candidate; final acceptance "
                        "requires no sequence uncertainty and no "
                        "overlap with ompF"
                    )

        decision = "missing"
        selected_hit = None

        if accepted_hit is not None:
            decision = "accepted"
            selected_hit = accepted_hit
        elif not locus_hits.empty:
            selected_hit = locus_hits.iloc[0]
            if (
                selected_hit["percent_identity"]
                >= BORDERLINE_IDENTITY
                and selected_hit["query_coverage"]
                >= BORDERLINE_COVERAGE
            ):
                decision = "borderline review"
            else:
                decision = "below review limits"

        if selected_hit is None:
            best_hit_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "locus_id": locus.locus_id,
                    "locus_name": locus.locus_name,
                    "decision": "no BLAST alignment",
                }
            )
            review_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "review_type":
                        "coding locus not recovered",
                    "locus_id": locus.locus_id,
                    "locus_name": locus.locus_name,
                    "details": "No BLAST alignment was found.",
                }
            )
            continue

        best_hit_record = {
            "biosample": biosample,
            "assembly_accession": assembly_accession,
            "locus_id": locus.locus_id,
            "locus_name": locus.locus_name,
            **{
                column: selected_hit[column]
                for column in BLAST_COLUMNS[1:]
            },
            "aligned_reference_bases":
                selected_hit["aligned_reference_bases"],
            "query_coverage":
                selected_hit["query_coverage"],
            "passes_threshold":
                selected_hit["passes_threshold"],
            "decision": decision,
            "acceptance_rule": acceptance_rule,
            "review_reason": review_reason,
        }
        best_hit_rows.append(best_hit_record)

        extracted = extract_reference_oriented_sequence(
            selected_hit,
            assembly_sequences,
        )
        integrity = assess_coding_integrity(
            extracted["sequence"],
            locus.reference_sequence,
            extracted["touches_contig_edge"],
        )

        accepted_for_kernel = (
            decision == "accepted"
            and integrity["coding_integrity"] != "uncertain"
        )
        sequence_status = (
            "sequence extracted"
            if accepted_for_kernel
            else (
                "uncertain"
                if (
                    decision == "accepted"
                    and integrity["coding_integrity"]
                    == "uncertain"
                )
                else decision
            )
        )

        sequence_rows.append(
            {
                "biosample": biosample,
                "assembly_accession": assembly_accession,
                "locus_id": locus.locus_id,
                "locus_name": locus.locus_name,
                "locus_group": locus.locus_group,
                "sequence_source":
                    "direct BLASTN against MG1655 reference",
                **extracted,
                "accepted_for_kernel":
                    accepted_for_kernel,
                "sequence_status": sequence_status,
                **integrity,
                "percent_identity":
                    float(
                        selected_hit["percent_identity"]
                    ),
                "query_coverage":
                    float(
                        selected_hit["query_coverage"]
                    ),
                "reference_length":
                    int(selected_hit["reference_length"]),
                "acceptance_rule": acceptance_rule,
                "review_reason": review_reason,
            }
        )

        if decision != "accepted":
            review_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "review_type":
                        "coding-locus alignment review",
                    "locus_id": locus.locus_id,
                    "locus_name": locus.locus_name,
                    "details": (
                        f"{decision}: "
                        f"{selected_hit['percent_identity']:.3f}% "
                        "identity; "
                        f"{selected_hit['query_coverage']:.3f}% "
                        "reference coverage"
                    ),
                }
            )

        if integrity["coding_integrity"] in {
            "potentially disrupted",
            "uncertain",
        }:
            review_rows.append(
                {
                    "biosample": biosample,
                    "assembly_accession":
                        assembly_accession,
                    "review_type": (
                        "potential coding disruption"
                        if integrity["coding_integrity"]
                        == "potentially disrupted"
                        else "uncertain coding sequence"
                    ),
                    "locus_id": locus.locus_id,
                    "locus_name": locus.locus_name,
                    "details": integrity["integrity_reason"],
                }
            )

    # ompC and ompF must not be assigned to overlapping genomic intervals.
    accepted_porins = [
        row
        for row in sequence_rows
        if (
            row["locus_name"] in {"ompC", "ompF"}
            and row["accepted_for_kernel"]
        )
    ]
    if len(accepted_porins) == 2:
        first, second = accepted_porins
        if first["contig_id"] == second["contig_id"]:
            overlap_start = max(
                int(first["start"]),
                int(second["start"]),
            )
            overlap_stop = min(
                int(first["stop"]),
                int(second["stop"]),
            )
            overlap_bases = max(
                0,
                overlap_stop - overlap_start + 1,
            )
            if overlap_bases > 0:
                for row in accepted_porins:
                    row["accepted_for_kernel"] = False
                    row["sequence_status"] = (
                        "ompC/ompF overlap requires review"
                    )
                    row["review_reason"] = (
                        f"ompC and ompF overlap by "
                        f"{overlap_bases} nucleotide(s)"
                    )
                review_rows.append(
                    {
                        "biosample": biosample,
                        "assembly_accession":
                            assembly_accession,
                        "review_type":
                            "ompC and ompF overlap",
                        "locus_id": "LOCUS_0281;LOCUS_0282",
                        "locus_name": "ompC;ompF",
                        "details": (
                            f"Accepted intervals overlap by "
                            f"{overlap_bases} nucleotide(s)."
                        ),
                    }
                )

    return sequence_rows, best_hit_rows, review_rows



def download_query_assembly(assembly_accession, assembly_directory):
    last_error = None
    for attempt in range(1, COMMAND_RETRIES + 1):
        if assembly_directory.exists():
            shutil.rmtree(assembly_directory)
        assembly_directory.mkdir(parents=True, exist_ok=True)
        archive_path = assembly_directory / f"{assembly_accession}.zip"
        try:
            result = subprocess.run([
                str(datasets_executable), "download", "genome", "accession",
                assembly_accession, "--include", "genome", "--filename", str(archive_path),
            ], capture_output=True, text=True, env=command_environment)
            if result.returncode != 0:
                raise RuntimeError(result.stderr.strip() or result.stdout.strip())
            if not zipfile.is_zipfile(archive_path):
                raise ValueError("The downloaded assembly is not a valid ZIP archive.")
            extracted_directory = assembly_directory / "genome"
            extracted_directory.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archive_path, "r") as archive:
                archive.extractall(extracted_directory)
            fasta_candidates = list(extracted_directory.rglob("*_genomic.fna"))
            if len(fasta_candidates) != 1:
                raise ValueError(
                    f"Expected one genomic FASTA for {assembly_accession}; found {len(fasta_candidates)}."
                )
            return fasta_candidates[0]
        except Exception as error:
            last_error = error
            if attempt < COMMAND_RETRIES:
                print(f"Assembly download attempt {attempt} failed; retrying in {5 * attempt} seconds.")
                time.sleep(5 * attempt)
    raise RuntimeError(
        f"Assembly {assembly_accession} failed after {COMMAND_RETRIES} attempts."
    ) from last_error


assembly_directory = ASSEMBLY_WORK_DIRECTORY / QUERY_ASSEMBLY_ACCESSION
assembly_fasta_path = download_query_assembly(QUERY_ASSEMBLY_ACCESSION, assembly_directory)
assembly_sequences = {
    record.id: str(record.seq).upper()
    for record in SeqIO.parse(str(assembly_fasta_path), "fasta")
}
if not assembly_sequences:
    raise ValueError(f"No genomic sequences were read for {QUERY_ASSEMBLY_ACCESSION}.")

query_amrfinder_report = run_amrfinder(
    QUERY_BIOSAMPLE,
    QUERY_ASSEMBLY_ACCESSION,
    assembly_fasta_path,
    assembly_directory,
)
amr_sequence_rows, amr_review_rows = extract_amrfinder_sequences(
    QUERY_BIOSAMPLE,
    QUERY_ASSEMBLY_ACCESSION,
    query_amrfinder_report,
    assembly_sequences,
)
coding_sequence_rows, query_coding_hit_rows, coding_review_rows = run_coding_reference_blast(
    QUERY_BIOSAMPLE,
    QUERY_ASSEMBLY_ACCESSION,
    assembly_fasta_path,
    assembly_sequences,
    assembly_directory,
)

query_sequence_records = pd.DataFrame(amr_sequence_rows + coding_sequence_rows)
query_review_items = pd.DataFrame(amr_review_rows + coding_review_rows)
query_coding_hits = pd.DataFrame(query_coding_hit_rows)

if query_sequence_records.empty:
    raise ValueError("No targeted sequence record was produced for the query assembly.")

query_sequence_records = query_sequence_records.sort_values([
    "assembly_accession", "locus_id", "contig_id", "start"
]).reset_index(drop=True)
query_sequence_records["copy_index"] = (
    query_sequence_records.groupby(["assembly_accession", "locus_id"]).cumcount() + 1
)
query_sequence_records["sequence_id"] = (
    query_sequence_records["biosample"] + "|"
    + query_sequence_records["assembly_accession"] + "|"
    + query_sequence_records["locus_id"] + "|copy_"
    + query_sequence_records["copy_index"].astype(str)
)

invalid_sequences = query_sequence_records[
    ~query_sequence_records["sequence"].astype(str).str.upper().map(
        lambda sequence: set(sequence) <= IUPAC_NUCLEOTIDES
    )
]
if not invalid_sequences.empty:
    raise ValueError("A query sequence contains an unexpected nucleotide symbol.")

QUERY_SEQUENCE_RECORD_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_targeted_sequence_records.csv.gz"
QUERY_REVIEW_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_targeted_sequence_review_items.csv"
QUERY_AMRFINDER_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_amrfinder_results.csv"
QUERY_CODING_HIT_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_coding_reference_blast_results.csv"

query_sequence_records.to_csv(QUERY_SEQUENCE_RECORD_PATH, index=False, compression="gzip")
query_review_items.to_csv(QUERY_REVIEW_PATH, index=False)
query_amrfinder_report.to_csv(QUERY_AMRFINDER_PATH, index=False)
query_coding_hits.to_csv(QUERY_CODING_HIT_PATH, index=False)

# The downloaded genome is no longer required after the extracted records are saved.
shutil.rmtree(assembly_directory)

extraction_summary = pd.DataFrame([
    {"metric": "Targeted locus panel", "value": EXPECTED_TARGETED_LOCI},
    {"metric": "Sequence records", "value": len(query_sequence_records)},
    {"metric": "Accepted sequence records", "value": int(query_sequence_records["accepted_for_kernel"].astype(bool).sum())},
    {"metric": "Unaccepted sequence records", "value": int((~query_sequence_records["accepted_for_kernel"].astype(bool)).sum())},
    {"metric": "Review records", "value": len(query_review_items)},
    {"metric": "Downloaded genome deleted", "value": not assembly_directory.exists()},
])
display(extraction_summary)

print(f"Saved: {QUERY_SEQUENCE_RECORD_PATH}")
print("\nTransition: Cell 30.10 will select one accepted query sequence per locus using the final Notebook 24 ranking rule.")


In [ ]:
#@title Cell 30.10 - Select one accepted query sequence per locus
# This cell reproduces Notebook 24's sequence-copy ranking and identifies which selected query loci have fixed training alignments.

accepted_mask = query_sequence_records["accepted_for_kernel"]
if not pd.api.types.is_bool_dtype(accepted_mask):
    accepted_mask = accepted_mask.astype(str).str.strip().str.lower().eq("true")
accepted_query_source = query_sequence_records.loc[accepted_mask].copy()

for numeric_column in ["query_coverage", "percent_identity", "copy_index"]:
    if numeric_column not in accepted_query_source.columns:
        accepted_query_source[numeric_column] = np.nan
    accepted_query_source[numeric_column] = pd.to_numeric(
        accepted_query_source[numeric_column], errors="coerce"
    )

accepted_query_source["sequence"] = accepted_query_source["sequence"].astype(str).str.upper()
accepted_query_source["sequence_length_for_ranking"] = accepted_query_source["sequence"].str.len()
accepted_query_source["coverage_for_ranking"] = accepted_query_source["query_coverage"].fillna(-1.0)
accepted_query_source["identity_for_ranking"] = accepted_query_source["percent_identity"].fillna(-1.0)
accepted_query_source["copy_for_ranking"] = accepted_query_source["copy_index"].fillna(np.inf)

accepted_query_source = accepted_query_source.sort_values(
    [
        "assembly_accession", "locus_id", "coverage_for_ranking",
        "identity_for_ranking", "sequence_length_for_ranking",
        "copy_for_ranking", "sequence_id",
    ],
    ascending=[True, True, False, False, False, True, True],
)

selected_query_sequences = (
    accepted_query_source.drop_duplicates(["assembly_accession", "locus_id"], keep="first")
    .copy()
    .reset_index(drop=True)
)
accepted_copy_counts = (
    accepted_query_source.groupby(["assembly_accession", "locus_id"]).size()
    .rename("accepted_copy_count")
    .reset_index()
)
selected_query_sequences = selected_query_sequences.merge(
    accepted_copy_counts,
    on=["assembly_accession", "locus_id"],
    how="left",
    validate="one_to_one",
)
selected_query_sequences["selection_rule"] = (
    "highest coverage, then highest identity, then longest sequence"
)
selected_query_sequences["alignment_record_id"] = "QUERY"

invalid_selected = selected_query_sequences[
    ~selected_query_sequences["sequence"].map(
        lambda sequence: bool(sequence) and set(sequence) <= {"A", "C", "G", "T"}
    )
]
if not invalid_selected.empty:
    raise ValueError("A selected query sequence contains an ambiguous nucleotide.")

represented_locus_ids = set(represented_alignment_table["locus_id"].astype(str))
selected_query_sequences["training_alignment_status"] = np.where(
    selected_query_sequences["locus_id"].isin(represented_locus_ids),
    "fixed training alignment available",
    "no fixed training alignment",
)

QUERY_SELECTED_SEQUENCE_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_selected_sequences.csv.gz"
selected_query_sequences.to_csv(QUERY_SELECTED_SEQUENCE_PATH, index=False, compression="gzip")

query_alignable_sequences = selected_query_sequences[
    selected_query_sequences["training_alignment_status"]
    == "fixed training alignment available"
].copy()

component5_summary = pd.DataFrame([{
    "component": "5",
    "name": "Targeted nucleotide sequences",
    "definition": "Selected query-locus nucleotide sequences compared within the corresponding fixed Notebook 24 alignments",
    "reported": EXPECTED_TARGETED_LOCI,
    "recognised": len(query_alignable_sequences),
    "unmatched": EXPECTED_TARGETED_LOCI - len(query_alignable_sequences),
    "example_values": ",".join(query_alignable_sequences["locus_name"].astype(str).head(5)),
}])
query_feature_summary = pd.concat([query_feature_summary, component5_summary], ignore_index=True)
query_feature_summary.to_csv(QUERY_FEATURE_SUMMARY_PATH, index=False)

sequence_selection_summary = pd.DataFrame([
    {"metric": "Targeted loci", "value": EXPECTED_TARGETED_LOCI},
    {"metric": "Accepted sequence records", "value": len(accepted_query_source)},
    {"metric": "Selected query loci", "value": len(selected_query_sequences)},
    {"metric": "Selected query loci with fixed training alignment", "value": len(query_alignable_sequences)},
    {"metric": "Selected query loci without fixed training alignment", "value": int((selected_query_sequences["training_alignment_status"] == "no fixed training alignment").sum())},
])
display(sequence_selection_summary)

print(f"Saved: {QUERY_SELECTED_SEQUENCE_PATH}")
print("\nTransition: Cell 30.11 will add each selected query sequence to its fixed training alignment and calculate the locus-weighted sequence similarity.")


In [ ]:
#@title Cell 30.11 - Calculate the locus-weighted query sequence similarity
# This restartable cell adds each query sequence to its unchanged training alignment and calculates one sequence-similarity value for every training pathogen.

QUERY_ALIGNMENT_CHECKPOINT_DIRECTORY = (
    NOTEBOOK30_CHECKPOINT_DIRECTORY / QUERY_BIOSAMPLE
)
QUERY_ALIGNMENT_CHECKPOINT_DIRECTORY.mkdir(parents=True, exist_ok=True)


def load_gzip_alignment(alignment_path):
    with gzip.open(alignment_path, "rt", encoding="utf-8") as input_file:
        return list(SeqIO.parse(input_file, "fasta"))


def sequence_sha256(sequence):
    return hashlib.sha256(str(sequence).upper().encode("ascii")).hexdigest()


raw_query_locus_similarity = np.zeros(EXPECTED_MODEL_C_PATHOGENS, dtype=np.float64)
query_locus_alignment_rows = []

for locus_number, query_locus in enumerate(
    query_alignable_sequences.sort_values("locus_id").itertuples(index=False),
    start=1,
):
    locus_id = str(query_locus.locus_id)
    query_sequence = str(query_locus.sequence).upper()
    training_alignment_path = (
        NOTEBOOK24_ALIGNMENT_DIRECTORY / f"24_{locus_id}_alignment.fasta.gz"
    )
    training_metadata_path = (
        NOTEBOOK24_ALIGNMENT_DIRECTORY / f"24_{locus_id}_alignment.json"
    )
    with open(training_metadata_path, "r", encoding="utf-8") as input_file:
        training_metadata = json.load(input_file)

    training_alignment_hash = file_sha256(training_alignment_path)
    if training_alignment_hash != training_metadata["alignment_sha256"]:
        raise ValueError(f"The fixed {locus_id} training alignment checksum does not match.")

    query_sequence_hash = sequence_sha256(query_sequence)
    checkpoint_path = QUERY_ALIGNMENT_CHECKPOINT_DIRECTORY / f"30_{locus_id}_query_similarity.npz"
    checkpoint_metadata_path = QUERY_ALIGNMENT_CHECKPOINT_DIRECTORY / f"30_{locus_id}_query_similarity.json"

    reuse_checkpoint = False
    if checkpoint_path.exists() and checkpoint_metadata_path.exists():
        with open(checkpoint_metadata_path, "r", encoding="utf-8") as input_file:
            checkpoint_metadata = json.load(input_file)
        reuse_checkpoint = (
            checkpoint_metadata.get("query_biosample") == QUERY_BIOSAMPLE
            and checkpoint_metadata.get("query_assembly_accession") == QUERY_ASSEMBLY_ACCESSION
            and checkpoint_metadata.get("query_sequence_sha256") == query_sequence_hash
            and checkpoint_metadata.get("training_alignment_sha256") == training_alignment_hash
            and checkpoint_metadata.get("normalisation")
            == "matching fixed aligned states divided by the fixed locus alignment length"
        )

    if reuse_checkpoint:
        with np.load(checkpoint_path) as checkpoint:
            training_row_indices = checkpoint["training_row_indices"].astype(np.int32)
            within_locus_similarities = checkpoint["within_locus_similarities"].astype(np.float64)
        aligned_length_bp = int(checkpoint_metadata["aligned_length_bp"])
        dropped_query_nucleotides = int(checkpoint_metadata["query_nucleotides_not_represented"])
        checkpoint_status = "existing checkpoint validated"
    else:
        if checkpoint_path.exists() != checkpoint_metadata_path.exists():
            raise FileNotFoundError(f"The query checkpoint for {locus_id} is incomplete.")

        training_records = load_gzip_alignment(training_alignment_path)
        expected_training_ids = [record.id for record in training_records]
        if len(expected_training_ids) != len(set(expected_training_ids)):
            raise ValueError(f"The fixed {locus_id} alignment contains duplicate identifiers.")
        if "ANCHOR" not in expected_training_ids:
            raise ValueError(f"The fixed {locus_id} alignment has no alignment reference sequence.")

        aligned_lengths = {len(record.seq) for record in training_records}
        if len(aligned_lengths) != 1:
            raise ValueError(f"The fixed {locus_id} alignment has unequal sequence lengths.")
        aligned_length_bp = int(next(iter(aligned_lengths)))

        local_training_path = ALIGNMENT_WORK_DIRECTORY / f"{locus_id}_training_alignment.fasta"
        local_query_path = ALIGNMENT_WORK_DIRECTORY / f"{locus_id}_query.fasta"
        local_output_path = ALIGNMENT_WORK_DIRECTORY / f"{locus_id}_query_added.fasta"

        with open(local_training_path, "w", encoding="utf-8") as output_file:
            SeqIO.write(training_records, output_file, "fasta")
        with open(local_query_path, "w", encoding="utf-8") as output_file:
            output_file.write(f">QUERY\n{query_sequence}\n")

        with open(local_output_path, "w", encoding="utf-8") as output_file:
            mafft_add_result = subprocess.run(
                [
                    MAFFT_EXECUTABLE,
                    "--nuc",
                    "--add",
                    str(local_query_path),
                    "--keeplength",
                    "--inputorder",
                    "--thread",
                    str(THREADS),
                    str(local_training_path),
                ],
                stdout=output_file,
                stderr=subprocess.PIPE,
                text=True,
                check=False,
            )
        if mafft_add_result.returncode != 0:
            raise RuntimeError(
                f"MAFFT query addition failed for {locus_id}:\n{mafft_add_result.stderr[-3000:]}"
            )

        added_records = list(SeqIO.parse(str(local_output_path), "fasta"))
        added_ids = [record.id for record in added_records]
        if set(added_ids) != {*expected_training_ids, "QUERY"}:
            raise ValueError(f"The {locus_id} query-added alignment contains unexpected identifiers.")
        if len(added_ids) != len(set(added_ids)):
            raise ValueError(f"The {locus_id} query-added alignment contains duplicate identifiers.")
        if {len(record.seq) for record in added_records} != {aligned_length_bp}:
            raise ValueError(f"The {locus_id} query-added alignment changed the fixed length.")

        original_training_sequences = {
            record.id: str(record.seq).upper() for record in training_records
        }
        output_lookup = {
            record.id: str(record.seq).upper() for record in added_records
        }
        changed_training_ids = [
            record_id for record_id, sequence in original_training_sequences.items()
            if output_lookup[record_id] != sequence
        ]
        if changed_training_ids:
            raise ValueError(
                f"MAFFT changed fixed training sequences for {locus_id}: {changed_training_ids[:5]}"
            )

        query_aligned_sequence = output_lookup["QUERY"]
        if not set(query_aligned_sequence) <= {"A", "C", "G", "T", "-"}:
            raise ValueError(f"The aligned query for {locus_id} contains an unexpected state.")
        represented_query_nucleotides = sum(
            nucleotide in {"A", "C", "G", "T"}
            for nucleotide in query_aligned_sequence
        )
        dropped_query_nucleotides = len(query_sequence) - represented_query_nucleotides
        if dropped_query_nucleotides < 0:
            raise ValueError(f"The aligned query for {locus_id} contains more bases than its input.")

        pathogen_records = [
            record for record in added_records
            if record.id.startswith("P")
        ]
        training_row_indices = np.array(
            [int(record.id[1:]) for record in pathogen_records],
            dtype=np.int32,
        )
        if (
            (training_row_indices < 0).any()
            or (training_row_indices >= EXPECTED_MODEL_C_PATHOGENS).any()
            or len(training_row_indices) != len(set(training_row_indices.tolist()))
        ):
            raise ValueError(f"The {locus_id} pathogen alignment identifiers are invalid.")

        training_bytes = np.frombuffer(
            "".join(str(record.seq).upper() for record in pathogen_records).encode("ascii"),
            dtype=np.uint8,
        ).reshape(len(pathogen_records), aligned_length_bp)
        query_bytes = np.frombuffer(query_aligned_sequence.encode("ascii"), dtype=np.uint8)
        within_locus_similarities = np.mean(
            training_bytes == query_bytes[None, :],
            axis=1,
            dtype=np.float64,
        )

        temporary_checkpoint_path = checkpoint_path.with_suffix(".npz.partial")
        with open(temporary_checkpoint_path, "wb") as output_file:
            np.savez_compressed(
                output_file,
                training_row_indices=training_row_indices,
                within_locus_similarities=within_locus_similarities.astype(np.float32),
            )
        temporary_checkpoint_path.replace(checkpoint_path)

        checkpoint_metadata = {
            "query_biosample": QUERY_BIOSAMPLE,
            "query_assembly_accession": QUERY_ASSEMBLY_ACCESSION,
            "locus_id": locus_id,
            "locus_name": str(query_locus.locus_name),
            "query_sequence_sha256": query_sequence_hash,
            "training_alignment_sha256": training_alignment_hash,
            "training_pathogen_sequences": int(len(training_row_indices)),
            "aligned_length_bp": aligned_length_bp,
            "query_sequence_length": int(len(query_sequence)),
            "query_nucleotides_not_represented": int(dropped_query_nucleotides),
            "normalisation": "matching fixed aligned states divided by the fixed locus alignment length",
            "mafft_version": mafft_version,
        }
        temporary_metadata_path = checkpoint_metadata_path.with_suffix(".json.partial")
        with open(temporary_metadata_path, "w", encoding="utf-8") as output_file:
            json.dump(checkpoint_metadata, output_file, indent=2)
        temporary_metadata_path.replace(checkpoint_metadata_path)

        for local_path in [local_training_path, local_query_path, local_output_path]:
            local_path.unlink(missing_ok=True)
        checkpoint_status = "saved and validated"

    if len(training_row_indices) != len(within_locus_similarities):
        raise ValueError(f"The {locus_id} checkpoint arrays have unequal lengths.")
    if not np.isfinite(within_locus_similarities).all():
        raise ValueError(f"The {locus_id} similarities contain invalid values.")
    if (
        (within_locus_similarities < -1e-8).any()
        or (within_locus_similarities > 1.0 + 1e-8).any()
    ):
        raise ValueError(f"The {locus_id} similarities are outside 0 to 1.")

    raw_query_locus_similarity[training_row_indices] += within_locus_similarities
    query_locus_alignment_rows.append({
        "locus_id": locus_id,
        "locus_name": str(query_locus.locus_name),
        "training_pathogen_sequences": len(training_row_indices),
        "aligned_length_bp": aligned_length_bp,
        "query_sequence_length": len(query_sequence),
        "query_nucleotides_not_represented": dropped_query_nucleotides,
        "checkpoint_status": checkpoint_status,
    })
    print(
        f"[{locus_number}/{len(query_alignable_sequences)}] {locus_id}: "
        f"{len(training_row_indices):,} training pathogens, "
        f"{aligned_length_bp:,} bp fixed alignment, {checkpoint_status}"
    )

query_locus_alignment_diagnostics = pd.DataFrame(query_locus_alignment_rows)
query_available_locus_count = len(query_locus_alignment_diagnostics)
if query_available_locus_count == 0:
    raise ValueError("The query has no accepted locus sequence represented by a fixed training alignment.")

denominator = np.sqrt(
    np.float64(query_available_locus_count)
    * training_available_locus_counts.astype(np.float64)
)
query_sequence_similarity = np.divide(
    raw_query_locus_similarity,
    denominator,
    out=np.zeros(EXPECTED_MODEL_C_PATHOGENS, dtype=np.float64),
    where=denominator > 0,
)

if query_sequence_similarity.shape != (EXPECTED_MODEL_C_PATHOGENS,):
    raise ValueError("The query sequence-similarity vector has the wrong dimension.")
if not np.isfinite(query_sequence_similarity).all():
    raise ValueError("The query sequence-similarity vector contains invalid values.")
if query_sequence_similarity.min() < -1e-6 or query_sequence_similarity.max() > 1.0 + 1e-6:
    raise ValueError("The query sequence-similarity values are outside 0 to 1.")

QUERY_LOCUS_DIAGNOSTIC_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_locus_alignment_diagnostics.csv"
QUERY_SEQUENCE_SIMILARITY_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_locus_weighted_sequence_similarity.npz"
query_locus_alignment_diagnostics.to_csv(QUERY_LOCUS_DIAGNOSTIC_PATH, index=False)
np.savez_compressed(
    QUERY_SEQUENCE_SIMILARITY_PATH,
    query_sequence_similarity=query_sequence_similarity.astype(np.float32),
    raw_locus_similarity_sum=raw_query_locus_similarity.astype(np.float32),
    query_available_locus_count=np.array([query_available_locus_count], dtype=np.int32),
)

sequence_similarity_preview = model_c_pathogen_index[
    ["model_c_row_index", "biosample", "assembly_accession"]
].copy()
sequence_similarity_preview["locus_weighted_sequence_similarity"] = query_sequence_similarity
display(sequence_similarity_preview.sort_values(
    "locus_weighted_sequence_similarity", ascending=False
).head(10))

print(f"Query loci contributing to sequence similarity: {query_available_locus_count}")
print(
    "Query nucleotides not represented by fixed training alignment positions: "
    f"{int(query_locus_alignment_diagnostics['query_nucleotides_not_represented'].sum())}"
)
print("\nTransition: Cell 30.12 will combine the Model 3B and sequence similarities and calculate prediction support.")


In [ ]:
#@title Cell 30.12 - Construct the Model C query similarity vector
# This cell applies rho=0.4, identifies the nearest training pathogen and compares the maximum Model C kernel element with the empirical support boundary.

query_model_c_similarity = (
    SELECTED_RHO * query_sequence_similarity
    + (1.0 - SELECTED_RHO) * model3b_similarity
)

if query_model_c_similarity.shape != (EXPECTED_MODEL_C_PATHOGENS,):
    raise ValueError("The Model C query similarity vector has the wrong dimension.")
if not np.isfinite(query_model_c_similarity).all():
    raise ValueError("The Model C query similarity vector contains invalid values.")
if query_model_c_similarity.min() < -1e-6 or query_model_c_similarity.max() > 1.0 + 1e-6:
    raise ValueError("The Model C query similarities are outside 0 to 1.")

EMPIRICAL_SUPPORT_BOUNDARY = float(support_configuration["empirical_support_boundary"])
nearest_training_row = int(np.argmax(query_model_c_similarity))
nearest_training_similarity = float(query_model_c_similarity[nearest_training_row])
nearest_training_biosample = str(model_c_pathogen_index.iloc[nearest_training_row]["biosample"])
nearest_training_assembly = str(model_c_pathogen_index.iloc[nearest_training_row]["assembly_accession"])
evaluated_similarity_status = (
    "inside evaluated similarity range"
    if nearest_training_similarity >= EMPIRICAL_SUPPORT_BOUNDARY
    else "outside evaluated similarity range"
)

query_model_c_similarity_table = model_c_pathogen_index[
    ["model_c_row_index", "model_3b_row_index", "biosample", "assembly_accession"]
].copy()
query_model_c_similarity_table["model3b_pathogen_similarity"] = model3b_similarity
query_model_c_similarity_table["locus_weighted_sequence_similarity"] = query_sequence_similarity
query_model_c_similarity_table["model_c_pathogen_similarity"] = query_model_c_similarity
query_model_c_similarity_table = query_model_c_similarity_table.sort_values(
    "model_c_pathogen_similarity", ascending=False
).reset_index(drop=True)

QUERY_MODEL_C_SIMILARITY_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_model_c_similarity_to_training_pathogens.csv.gz"
QUERY_TOP_MATCH_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_top_10_training_pathogens.csv"
query_model_c_similarity_table.to_csv(
    QUERY_MODEL_C_SIMILARITY_PATH, index=False, compression="gzip"
)
query_model_c_similarity_table.head(10).to_csv(QUERY_TOP_MATCH_PATH, index=False)

support_summary = pd.DataFrame([
    {"metric": "Nearest training pathogen", "value": nearest_training_biosample},
    {"metric": "Nearest training assembly", "value": nearest_training_assembly},
    {"metric": "Nearest-training-pathogen similarity", "value": nearest_training_similarity},
    {"metric": "Empirical support boundary", "value": EMPIRICAL_SUPPORT_BOUNDARY},
    {"metric": "Evaluated similarity status", "value": evaluated_similarity_status},
])
display(support_summary)
print("\nTen nearest training pathogens")
display(query_model_c_similarity_table.head(10))

print("\nThe scalar nearest-training-pathogen similarity is the largest element in the 9,058-element Model C query similarity vector.")
print("\nTransition: Cell 30.13 will project this vector into 256 pathogen coordinates and predict MIC for 26 antibiotics.")


In [ ]:
#@title Cell 30.13 - Project the query and predict 26 MIC values
# This cell converts the Model C similarity vector into 256 coordinates, forms 6,656 interaction features per antibiotic and applies the fixed Ridge model.

query_pathogen_coordinates = (
    query_model_c_similarity @ pathogen_embedding
) / pathogen_eigenvalues
query_pathogen_coordinates = np.asarray(query_pathogen_coordinates, dtype=np.float64).reshape(
    EXPECTED_PATHOGEN_COORDINATES
)
if not np.isfinite(query_pathogen_coordinates).all():
    raise ValueError("The query pathogen coordinates contain invalid values.")

approximated_query_similarity = pathogen_embedding @ query_pathogen_coordinates
coordinate_approximation_rmse = float(np.sqrt(np.mean(
    (query_model_c_similarity - approximated_query_similarity) ** 2
)))

query_interaction_features = np.einsum(
    "r,js->jrs",
    query_pathogen_coordinates,
    antibiotic_embedding,
    optimize=True,
).reshape(EXPECTED_ANTIBIOTICS, EXPECTED_INTERACTION_FEATURES)

if query_interaction_features.shape != (EXPECTED_ANTIBIOTICS, EXPECTED_INTERACTION_FEATURES):
    raise ValueError("The query interaction-feature matrix has the wrong dimension.")
if not np.isfinite(query_interaction_features).all():
    raise ValueError("The query interaction features contain invalid values.")

predicted_log2_mic = np.asarray(
    final_model_c.predict(query_interaction_features), dtype=np.float64
).reshape(-1)
if predicted_log2_mic.shape != (EXPECTED_ANTIBIOTICS,):
    raise ValueError("The final model did not return 26 predictions.")
if not np.isfinite(predicted_log2_mic).all():
    raise ValueError("The MIC predictions contain invalid values.")

observed_antibiotic_ranges = (
    model_c_interactions.groupby("antibiotic")["log2_mic"]
    .agg(observed_minimum_log2_mic="min", observed_maximum_log2_mic="max")
    .reset_index()
)

query_predictions = model_c_antibiotic_index[
    ["antibiotic_embedding_row", "antibiotic"]
].copy()
query_predictions["biosample"] = QUERY_BIOSAMPLE
query_predictions["assembly_accession"] = QUERY_ASSEMBLY_ACCESSION
query_predictions["predicted_log2_mic"] = predicted_log2_mic
query_predictions = query_predictions.merge(
    observed_antibiotic_ranges,
    on="antibiotic",
    how="left",
    validate="one_to_one",
)
if query_predictions[["observed_minimum_log2_mic", "observed_maximum_log2_mic"]].isna().any().any():
    raise ValueError("An observed antibiotic MIC range is missing.")

QUERY_COORDINATE_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_model_c_coordinates_and_features.npz"
np.savez_compressed(
    QUERY_COORDINATE_PATH,
    pathogen_coordinates=query_pathogen_coordinates,
    interaction_features=query_interaction_features,
)

print("Central Model C MIC predictions were calculated for all 26 antibiotics.")
print(f"256-coordinate similarity approximation RMSE: {coordinate_approximation_rmse:.6f}")
print("\nTransition: Cell 30.14 will attach the fixed 95% prediction limits and prediction-support information.")


In [ ]:
#@title Cell 30.14 - Attach 95% prediction limits and support information
# This cell adds the Notebook 28 antibiotic-specific residual limits, ordinary MIC units and the Notebook 27 prediction-support status.

required_interval_columns = [
    "antibiotic",
    "lower_residual_2_5_percentile",
    "upper_residual_97_5_percentile",
]
missing_interval_columns = [
    column for column in required_interval_columns
    if column not in prediction_interval_calibration.columns
]
if missing_interval_columns:
    raise ValueError(f"The prediction-interval table is missing {missing_interval_columns}.")

query_predictions = query_predictions.merge(
    prediction_interval_calibration[required_interval_columns],
    on="antibiotic",
    how="left",
    validate="one_to_one",
)
if query_predictions[required_interval_columns[1:]].isna().any().any():
    raise ValueError("A prediction-interval calibration value is missing.")

query_predictions["lower_95_prediction_limit_log2_mic"] = (
    query_predictions["predicted_log2_mic"]
    + query_predictions["lower_residual_2_5_percentile"]
)
query_predictions["upper_95_prediction_limit_log2_mic"] = (
    query_predictions["predicted_log2_mic"]
    + query_predictions["upper_residual_97_5_percentile"]
)
query_predictions["predicted_mic_mg_l"] = np.power(2.0, query_predictions["predicted_log2_mic"])
query_predictions["lower_95_prediction_limit_mic_mg_l"] = np.power(
    2.0, query_predictions["lower_95_prediction_limit_log2_mic"]
)
query_predictions["upper_95_prediction_limit_mic_mg_l"] = np.power(
    2.0, query_predictions["upper_95_prediction_limit_log2_mic"]
)
query_predictions["nearest_training_pathogen"] = nearest_training_biosample
query_predictions["nearest_training_pathogen_similarity"] = nearest_training_similarity
query_predictions["empirical_support_boundary"] = EMPIRICAL_SUPPORT_BOUNDARY
query_predictions["evaluated_similarity_status"] = evaluated_similarity_status
query_predictions["outside_observed_antibiotic_mic_range"] = (
    (query_predictions["predicted_log2_mic"] < query_predictions["observed_minimum_log2_mic"])
    | (query_predictions["predicted_log2_mic"] > query_predictions["observed_maximum_log2_mic"])
)
query_predictions["interpretation"] = "research prediction; confirmatory susceptibility testing required"

numeric_columns = [
    "predicted_log2_mic",
    "lower_95_prediction_limit_log2_mic",
    "upper_95_prediction_limit_log2_mic",
    "predicted_mic_mg_l",
    "lower_95_prediction_limit_mic_mg_l",
    "upper_95_prediction_limit_mic_mg_l",
]
if not np.isfinite(query_predictions[numeric_columns].to_numpy(dtype=float)).all():
    raise ValueError("The prediction table contains invalid numerical values.")
if (
    query_predictions["lower_95_prediction_limit_log2_mic"]
    > query_predictions["predicted_log2_mic"]
).any() or (
    query_predictions["upper_95_prediction_limit_log2_mic"]
    < query_predictions["predicted_log2_mic"]
).any():
    raise ValueError("At least one central prediction is outside its 95% prediction limits.")

QUERY_PREDICTION_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_new_pathogen_model_c_mic_predictions.csv"
query_predictions.to_csv(QUERY_PREDICTION_PATH, index=False)

display_columns = [
    "antibiotic",
    "predicted_log2_mic",
    "lower_95_prediction_limit_log2_mic",
    "upper_95_prediction_limit_log2_mic",
    "predicted_mic_mg_l",
    "lower_95_prediction_limit_mic_mg_l",
    "upper_95_prediction_limit_mic_mg_l",
    "evaluated_similarity_status",
    "outside_observed_antibiotic_mic_range",
]
display(query_predictions[display_columns].round(4))

print(f"Saved: {QUERY_PREDICTION_PATH}")
print("\nThe 95% prediction limits are post-model calculations and do not change the central Model C prediction.")
print("\nTransition: Cell 30.15 will create the final quality-control, configuration and Excel outputs.")


In [ ]:
#@title Cell 30.15 - Create final quality-control and readable outputs
# This cell summarises feature, sequence, similarity and prediction support and saves one readable Excel workbook plus the complete configuration.

QUERY_QUALITY_CONTROL_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_prediction_quality_control.csv"
QUERY_CONFIGURATION_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_query_model_c_prediction_configuration.json"
QUERY_EXCEL_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_new_pathogen_model_c_mic_predictions.xlsx"

query_quality_control = pd.DataFrame([
    {"check": "Query BioSample", "value": QUERY_BIOSAMPLE},
    {"check": "Query assembly accession", "value": QUERY_ASSEMBLY_ACCESSION},
    {"check": "Recognised Component 1 features", "value": int(query_full_raw.sum())},
    {"check": "Recognised Component 2 features", "value": int(query_core_raw.sum())},
    {"check": "Recognised Component 3 target loci", "value": int(query_point_raw.sum())},
    {"check": "Component 4 numerical features", "value": 5},
    {"check": "Component 5 query loci contributing", "value": query_available_locus_count},
    {"check": "Query nucleotides not represented by fixed positions", "value": int(query_locus_alignment_diagnostics["query_nucleotides_not_represented"].sum())},
    {"check": "Nearest training pathogen", "value": nearest_training_biosample},
    {"check": "Nearest-training-pathogen similarity", "value": nearest_training_similarity},
    {"check": "Empirical support boundary", "value": EMPIRICAL_SUPPORT_BOUNDARY},
    {"check": "Evaluated similarity status", "value": evaluated_similarity_status},
    {"check": "256-coordinate approximation RMSE", "value": coordinate_approximation_rmse},
    {"check": "Predicted antibiotics", "value": len(query_predictions)},
    {"check": "Predictions outside observed antibiotic MIC range", "value": int(query_predictions["outside_observed_antibiotic_mic_range"].sum())},
    {"check": "Nested pathogen-out Model C MAE", "value": manifest26["nested_model_c_performance"]["mae"]},
    {"check": "Nested pathogen-out Model C RMSE", "value": manifest26["nested_model_c_performance"]["rmse"]},
    {"check": "Nested pathogen-out Model C Pearson r", "value": manifest26["nested_model_c_performance"]["pearson_r"]},
])
query_quality_control.to_csv(QUERY_QUALITY_CONTROL_PATH, index=False)

query_configuration = {
    "notebook": 30,
    "model": "Model C",
    "query_biosample": QUERY_BIOSAMPLE,
    "query_assembly_accession": QUERY_ASSEMBLY_ACCESSION,
    "model_c_training_pathogens": EXPECTED_MODEL_C_PATHOGENS,
    "targeted_loci": EXPECTED_TARGETED_LOCI,
    "query_loci_contributing_to_sequence_similarity": query_available_locus_count,
    "selected_sequence_kernel": SELECTED_SEQUENCE_KERNEL,
    "selected_rho": SELECTED_RHO,
    "combined_similarity_formula": "0.4*K_seq(q,j) + 0.6*K_P_3B(q,j)",
    "nearest_training_pathogen": nearest_training_biosample,
    "nearest_training_pathogen_similarity": nearest_training_similarity,
    "empirical_support_boundary": EMPIRICAL_SUPPORT_BOUNDARY,
    "evaluated_similarity_status": evaluated_similarity_status,
    "pathogen_coordinates": EXPECTED_PATHOGEN_COORDINATES,
    "antibiotic_coordinates": EXPECTED_ANTIBIOTIC_COORDINATES,
    "interaction_features": EXPECTED_INTERACTION_FEATURES,
    "ridge_alpha": SELECTED_RIDGE_ALPHA,
    "prediction_interval_method": (
        "For each antibiotic, add the 2.5th and 97.5th percentiles of "
        "independent nested pathogen-out residuals to the central prediction."
    ),
    "fixed_alignment_rule": (
        "The query is added with MAFFT --add --keeplength; the 9,058-pathogen "
        "training alignment positions are unchanged."
    ),
    "query_nucleotides_not_represented": int(
        query_locus_alignment_diagnostics["query_nucleotides_not_represented"].sum()
    ),
    "clinical_breakpoint_interpretation": "not performed",
    "validation_status": "passed",
}
with open(QUERY_CONFIGURATION_PATH, "w", encoding="utf-8") as output_file:
    json.dump(query_configuration, output_file, indent=2)

with pd.ExcelWriter(QUERY_EXCEL_PATH, engine="openpyxl") as excel_writer:
    query_predictions.to_excel(excel_writer, sheet_name="MIC_predictions", index=False)
    query_quality_control.to_excel(excel_writer, sheet_name="Quality_control", index=False)
    query_feature_summary.to_excel(excel_writer, sheet_name="Five_components", index=False)
    query_model_c_similarity_table.head(25).to_excel(excel_writer, sheet_name="Nearest_pathogens", index=False)
    query_locus_alignment_diagnostics.to_excel(excel_writer, sheet_name="Sequence_alignment", index=False)
    software_versions.to_excel(excel_writer, sheet_name="Software_versions", index=False)

with zipfile.ZipFile(QUERY_EXCEL_PATH, "r") as excel_archive:
    damaged_member = excel_archive.testzip()
if damaged_member is not None:
    raise ValueError(f"The Excel workbook contains a damaged member: {damaged_member}")

display(query_quality_control)
print(f"Saved: {QUERY_EXCEL_PATH}")
print("\nTransition: Cell 30.16 will create the output manifest, package all validated query files and report the final status.")


In [ ]:
#@title Cell 30.16 - Package and report the final Notebook 30 outputs
# This cell records checksums, creates one validated ZIP archive and reports whether the complete new-pathogen Model C prediction passed validation.

OUTPUT_MANIFEST_PATH = NOTEBOOK30_RESULT_DIRECTORY / "30_output_manifest.json"
FINAL_OUTPUT_ARCHIVE_PATH = (
    NOTEBOOK30_DIRECTORY / f"30_{QUERY_BIOSAMPLE}_new_pathogen_model_c_mic_prediction_outputs.zip"
)

files_to_package = [
    QUERY_METADATA_PATH,
    QUERY_FEATURE_SUMMARY_PATH,
    UNMATCHED_FULL_PATH,
    UNMATCHED_CORE_PATH,
    QUERY_FEATURE_VECTOR_PATH,
    MODEL3B_SIMILARITY_PATH,
    SOFTWARE_VERSION_PATH,
    QUERY_SEQUENCE_RECORD_PATH,
    QUERY_REVIEW_PATH,
    QUERY_AMRFINDER_PATH,
    QUERY_CODING_HIT_PATH,
    QUERY_SELECTED_SEQUENCE_PATH,
    QUERY_LOCUS_DIAGNOSTIC_PATH,
    QUERY_SEQUENCE_SIMILARITY_PATH,
    QUERY_MODEL_C_SIMILARITY_PATH,
    QUERY_TOP_MATCH_PATH,
    QUERY_COORDINATE_PATH,
    QUERY_PREDICTION_PATH,
    QUERY_QUALITY_CONTROL_PATH,
    QUERY_CONFIGURATION_PATH,
    QUERY_EXCEL_PATH,
]

missing_output_files = [path for path in files_to_package if not path.exists()]
if missing_output_files:
    raise FileNotFoundError(f"Notebook 30 output files are missing: {missing_output_files}")

output_manifest = {
    "notebook": 30,
    "model": "Model C",
    "query_biosample": QUERY_BIOSAMPLE,
    "query_assembly_accession": QUERY_ASSEMBLY_ACCESSION,
    "predicted_antibiotics": len(query_predictions),
    "predictions_with_95_percent_prediction_limits": int(
        query_predictions["lower_95_prediction_limit_log2_mic"].notna().sum()
    ),
    "nearest_training_pathogen_similarity": nearest_training_similarity,
    "empirical_support_boundary": EMPIRICAL_SUPPORT_BOUNDARY,
    "evaluated_similarity_status": evaluated_similarity_status,
    "selected_sequence_kernel": SELECTED_SEQUENCE_KERNEL,
    "selected_rho": SELECTED_RHO,
    "selected_pathogen_dimension": EXPECTED_PATHOGEN_COORDINATES,
    "selected_ridge_alpha": SELECTED_RIDGE_ALPHA,
    "files": [
        {
            "file_name": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        }
        for path in files_to_package
    ],
    "validation_status": "passed",
}

temporary_manifest_path = OUTPUT_MANIFEST_PATH.with_suffix(".json.partial")
with open(temporary_manifest_path, "w", encoding="utf-8") as output_file:
    json.dump(output_manifest, output_file, indent=2)
temporary_manifest_path.replace(OUTPUT_MANIFEST_PATH)
files_to_package.append(OUTPUT_MANIFEST_PATH)

local_archive_path = WORK_DIRECTORY / FINAL_OUTPUT_ARCHIVE_PATH.name
local_archive_path.unlink(missing_ok=True)
with zipfile.ZipFile(
    local_archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
) as archive:
    for file_path in files_to_package:
        archive.write(file_path, arcname=file_path.name)

with zipfile.ZipFile(local_archive_path, "r") as archive:
    damaged_member = archive.testzip()
    archived_members = set(archive.namelist())
if damaged_member is not None:
    raise ValueError(f"The final archive contains a damaged member: {damaged_member}")
if archived_members != {path.name for path in files_to_package}:
    raise ValueError("The final archive member list is incomplete.")

partial_archive_path = FINAL_OUTPUT_ARCHIVE_PATH.with_suffix(".zip.partial")
partial_archive_path.unlink(missing_ok=True)
shutil.copy2(local_archive_path, partial_archive_path)
if file_sha256(partial_archive_path) != file_sha256(local_archive_path):
    raise IOError("The copied final archive does not match the validated local archive.")
partial_archive_path.replace(FINAL_OUTPUT_ARCHIVE_PATH)

final_summary = pd.DataFrame([
    {"metric": "Query BioSample", "value": QUERY_BIOSAMPLE},
    {"metric": "Query assembly", "value": QUERY_ASSEMBLY_ACCESSION},
    {"metric": "Sequence loci contributing", "value": query_available_locus_count},
    {"metric": "Nearest-training-pathogen similarity", "value": nearest_training_similarity},
    {"metric": "Empirical support boundary", "value": EMPIRICAL_SUPPORT_BOUNDARY},
    {"metric": "Evaluated similarity status", "value": evaluated_similarity_status},
    {"metric": "Predicted antibiotics", "value": len(query_predictions)},
    {"metric": "Predictions with 95% prediction limits", "value": int(query_predictions["lower_95_prediction_limit_log2_mic"].notna().sum())},
    {"metric": "Notebook 30 validation status", "value": "passed"},
])
display(final_summary)

print(f"Saved: {FINAL_OUTPUT_ARCHIVE_PATH}")
print(
    "\nNotebook 30 completed successfully. The new pathogen has 26 Model C "
    "MIC predictions, antibiotic-specific 95% prediction limits and explicit "
    "prediction-support information."
)
print("The Project C computational notebook series is complete.")
